## 공통 설정 (최초 1회 실행)

> 아래 셀을 **가장 먼저** 실행하세요. 이후 각 분석 셀을 독립적으로 실행할 수 있습니다.

In [ ]:
import numpy as np
import json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from skimage.measure import EllipseModel, CircleModel, ransac
from scipy import optimize
from scipy import stats
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# ============================================================================
# 데이터 로딩
# ============================================================================

class TreeDataLoader:
    def __init__(self, root_dir, label_csv=None):
        self.root_dir = Path(root_dir)
        if label_csv is None:
            self._label_path = self.root_dir / 'taehee' / 'label_filtered_f.csv'
        else:
            self._label_path = self.root_dir / label_csv
        
    def load_frame(self, frame_id):
        points_path = self.root_dir / 'clustered_dataset' / f'points_{frame_id:04d}.bin'
        if not points_path.exists():
            points_path = self.root_dir.parent / 'clustered_dataset' / f'points_{frame_id:04d}.bin'
        points = np.fromfile(points_path, dtype=np.float32).reshape(-1, 3)
        
        imu_path = self.root_dir / 'raw_dataset_filtered' / f'IMU_{frame_id}.json'
        with open(imu_path) as f:
            imu = json.load(f)
        
        df = pd.read_csv(self._label_path)
        row = df[df['frame'] == frame_id].iloc[0]
        
        return {
            'points': points,
            'imu': imu,
            'label': row,
            'actual_dbh': row['dbh']
        }


# ============================================================================
# 정렬 함수들 (IMU / PCA)
# ============================================================================

def align_with_imu(points, imu):
    """
    방법 1: IMU 회전행렬로 정렬
    - IMU의 회전행렬은 카메라→월드 변환
    - 월드 좌표계에서 Y축이 위쪽(중력 반대)
    """
    rot_matrix = np.array(imu['rotation_matrix'])
    aligned = points @ rot_matrix.T
    
    # 중심을 원점으로 이동
    centroid = aligned.mean(axis=0)
    aligned = aligned - centroid
    
    return aligned, rot_matrix, centroid


def align_with_pca(points, target_up=np.array([0, 1, 0])):
    """
    방법 2: PCA로 주축 찾아서 정렬
    - 나무의 주축(가장 큰 분산 방향)을 Y축과 정렬
    """
    # 중심 이동
    centroid = points.mean(axis=0)
    centered = points - centroid
    
    # PCA
    cov = np.cov(centered.T)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # 내림차순 정렬
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # 가장 큰 고유벡터 = 나무의 주축 (세로 방향)
    trunk_axis = eigenvectors[:, 0]
    
    # 주축이 위쪽을 향하도록 (Y+ 방향)
    if trunk_axis @ target_up < 0:
        trunk_axis = -trunk_axis
    
    # 회전 행렬 계산: trunk_axis → target_up
    rotation_matrix = rotation_matrix_from_vectors(trunk_axis, target_up)
    
    # 정렬
    aligned = centered @ rotation_matrix.T
    
    return aligned, rotation_matrix, centroid, trunk_axis, eigenvalues, eigenvectors


def rotation_matrix_from_vectors(vec1, vec2):
    """
    vec1을 vec2로 회전시키는 회전행렬 계산 (Rodrigues' formula)
    """
    a = vec1 / np.linalg.norm(vec1)
    b = vec2 / np.linalg.norm(vec2)
    
    v = np.cross(a, b)
    c = np.dot(a, b)
    
    if np.linalg.norm(v) < 1e-8:
        # 평행하거나 반평행
        if c > 0:
            return np.eye(3)
        else:
            # 180도 회전
            ortho = np.array([1, 0, 0]) if abs(a[0]) < 0.9 else np.array([0, 1, 0])
            ortho = ortho - np.dot(ortho, a) * a
            ortho = ortho / np.linalg.norm(ortho)
            return 2 * np.outer(ortho, ortho) - np.eye(3)
    
    s = np.linalg.norm(v)
    kmat = np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])
    
    rotation_matrix = np.eye(3) + kmat + kmat @ kmat * ((1 - c) / (s ** 2))
    return rotation_matrix


def combined_alignment(points, imu, method='imu_then_pca'):
    """
    방법 3: IMU + PCA 조합
    - IMU로 대략적 정렬 후 PCA로 미세 조정
    """
    if method == 'imu_only':
        aligned, rot, centroid = align_with_imu(points, imu)
        return aligned, {'method': 'imu_only', 'rotation': rot, 'centroid': centroid}
    
    elif method == 'pca_only':
        aligned, rot, centroid, axis, eig_vals, eig_vecs = align_with_pca(points)
        return aligned, {
            'method': 'pca_only', 
            'rotation': rot, 
            'centroid': centroid,
            'trunk_axis': axis,
            'eigenvalues': eig_vals,
            'eigenvectors': eig_vecs
        }
    
    elif method == 'imu_then_pca':
        # 1단계: IMU로 정렬
        aligned_imu, rot_imu, centroid_imu = align_with_imu(points, imu)
        
        # 2단계: PCA로 미세 조정
        aligned_pca, rot_pca, centroid_pca, axis, eig_vals, eig_vecs = align_with_pca(aligned_imu)
        
        return aligned_pca, {
            'method': 'imu_then_pca',
            'rotation_imu': rot_imu,
            'rotation_pca': rot_pca,
            'centroid': centroid_imu,
            'trunk_axis': axis,
            'eigenvalues': eig_vals
        }
    
    else:
        raise ValueError(f"Unknown method: {method}")


# ============================================================================
# 슬라이스 및 피팅
# ============================================================================

def extract_horizontal_slice(points, height, thickness=0.02):
    """
    수평 슬라이스 추출 (Y축 = 높이)
    - 정렬 후에는 Y축이 나무의 세로 방향
    """
    y_vals = points[:, 1]
    mask = np.abs(y_vals - height) < thickness / 2
    
    # XZ 평면 (수평 단면)
    slice_points = points[mask]
    xz = slice_points[:, [0, 2]]  # X, Z 좌표
    
    return xz, slice_points


def extract_slice_by_ratio(points, height_ratio=0.5, thickness=0.02):
    """
    높이 비율로 슬라이스 추출
    """
    y_vals = points[:, 1]
    y_min, y_max = y_vals.min(), y_vals.max()
    target_height = y_min + (y_max - y_min) * height_ratio
    
    return extract_horizontal_slice(points, target_height, thickness)


def filter_noise_dbscan(xz, eps=0.05, min_samples=5):
    """DBSCAN 노이즈 제거"""
    if len(xz) < min_samples:
        return xz, np.zeros(len(xz), dtype=int)
    
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(xz)
    labels = clustering.labels_
    
    unique_labels = set(labels)
    if -1 in unique_labels:
        unique_labels.remove(-1)
    
    if not unique_labels:
        return xz, labels
    
    largest_cluster = max(unique_labels, key=lambda l: np.sum(labels == l))
    mask = labels == largest_cluster
    return xz[mask], labels


# ============================================================================
# PCA 투영 방식 (새 코드에서 가져옴)
# ============================================================================

def pca_projection_extract(points, up=np.array([0, 1, 0], dtype=float)):
    """
    PCA 기반 투영 방식으로 2D 단면 추출
    - 점을 회전하지 않고 PCA 축을 찾아서 그 평면에 투영
    - 모든 점을 사용 (슬라이스 없음)
    
    Returns:
        points_2d: 투영된 2D 점들
        pca_info: PCA 관련 정보 (axis, mu, u, v 등)
    """
    # PCA
    mu = points.mean(axis=0)
    X = points - mu
    C = (X.T @ X) / len(points)
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    
    # 내림차순 정렬
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # up과 가장 정렬된 축 선택 (나무 주축)
    dots = np.abs(eigenvectors.T @ up)
    axis_idx = np.argmax(dots)
    axis = eigenvectors[:, axis_idx]
    axis = axis / np.linalg.norm(axis)
    if axis @ up < 0:
        axis = -axis
    
    # 단면 좌표계 (u, v) - 주축에 수직인 평면
    tmp = up if abs(axis @ up) < 0.95 else np.array([1, 0, 0])
    u = np.cross(axis, tmp)
    u = u / np.linalg.norm(u)
    v = np.cross(axis, u)
    
    # 2D로 투영
    diff = points - mu
    points_2d = np.stack([diff @ u, diff @ v], axis=1)
    
    return points_2d, {
        'axis': axis,
        'mu': mu,
        'u': u,
        'v': v,
        'eigenvalues': eigenvalues,
        'eigenvectors': eigenvectors
    }


def pca_projection_with_slice(points, up=np.array([0, 1, 0], dtype=float), 
                               height_ratio=0.5, thickness=0.02):
    """
    PCA 투영 + 높이 슬라이스 결합
    - 먼저 PCA로 주축을 찾고
    - 주축 방향으로 슬라이스한 후
    - 그 점들을 u, v 평면에 투영
    
    Returns:
        points_2d: 투영된 2D 점들 (슬라이스된)
        slice_3d: 슬라이스된 3D 점들
        pca_info: PCA 관련 정보
    """
    # PCA
    mu = points.mean(axis=0)
    X = points - mu
    C = (X.T @ X) / len(points)
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # 주축 찾기
    dots = np.abs(eigenvectors.T @ up)
    axis_idx = np.argmax(dots)
    axis = eigenvectors[:, axis_idx]
    axis = axis / np.linalg.norm(axis)
    if axis @ up < 0:
        axis = -axis
    
    # 단면 좌표계
    tmp = up if abs(axis @ up) < 0.95 else np.array([1, 0, 0])
    u = np.cross(axis, tmp)
    u = u / np.linalg.norm(u)
    v = np.cross(axis, u)
    
    # 주축 방향 높이 계산
    diff = points - mu
    heights = diff @ axis  # 주축 방향 거리
    
    # 슬라이스
    h_min, h_max = heights.min(), heights.max()
    target_height = h_min + (h_max - h_min) * height_ratio
    mask = np.abs(heights - target_height) < thickness / 2
    
    slice_3d = points[mask]
    diff_slice = slice_3d - mu
    
    # 2D로 투영
    points_2d = np.stack([diff_slice @ u, diff_slice @ v], axis=1)
    
    return points_2d, slice_3d, {
        'axis': axis,
        'mu': mu,
        'u': u,
        'v': v,
        'eigenvalues': eigenvalues,
        'eigenvectors': eigenvectors,
        'target_height': target_height
    }


# ============================================================================
# 기울기 보정 없는 방식 (No Alignment)
# ============================================================================

def extract_slice_no_alignment(points, height_ratio=0.5, thickness=0.02):
    """
    기울기 보정 없이 원본 점군에서 바로 슬라이스
    - Y축을 높이로 가정하고 슬라이스
    - XZ 평면 추출
    """
    # 중심 이동만
    centroid = points.mean(axis=0)
    centered = points - centroid
    
    # Y축 기준 슬라이스 (보정 없음)
    y_vals = centered[:, 1]
    y_min, y_max = y_vals.min(), y_vals.max()
    target_height = y_min + (y_max - y_min) * height_ratio
    
    mask = np.abs(y_vals - target_height) < thickness / 2
    slice_3d = centered[mask]
    
    # XZ 평면 추출
    xz = slice_3d[:, [0, 2]]
    
    return xz, slice_3d, {
        'method': 'no_alignment',
        'centroid': centroid,
        'target_height': target_height
    }


def extract_slice_no_alignment_z_axis(points, height_ratio=0.5, thickness=0.02):
    """
    기울기 보정 없이 Z축을 높이로 가정하고 슬라이스
    - XY 평면 추출
    """
    centroid = points.mean(axis=0)
    centered = points - centroid
    
    # Z축 기준 슬라이스
    z_vals = centered[:, 2]
    z_min, z_max = z_vals.min(), z_vals.max()
    target_height = z_min + (z_max - z_min) * height_ratio
    
    mask = np.abs(z_vals - target_height) < thickness / 2
    slice_3d = centered[mask]
    
    # XY 평면 추출
    xy = slice_3d[:, [0, 1]]
    
    return xy, slice_3d, {
        'method': 'no_alignment_z',
        'centroid': centroid,
        'target_height': target_height
    }


# ============================================================================
# 방법 1: 중심선(Centerline) 기반 정렬
# ============================================================================

def extract_centerline(points, n_slices=20, up=np.array([0, 1, 0])):
    """
    여러 슬라이스의 중심점을 연결하여 중심선 추출
    
    Returns:
        centerline_points: 각 슬라이스의 중심점들
        centerline_direction: 중심선의 방향벡터 (선형 피팅)
        slice_info: 각 슬라이스 정보
    """
    centroid = points.mean(axis=0)
    centered = points - centroid
    
    # PCA로 대략적인 주축 찾기
    cov = np.cov(centered.T)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvectors = eigenvectors[:, idx]
    
    # up과 가장 정렬된 축
    dots = np.abs(eigenvectors.T @ up)
    axis_idx = np.argmax(dots)
    rough_axis = eigenvectors[:, axis_idx]
    if rough_axis @ up < 0:
        rough_axis = -rough_axis
    
    # 주축 방향 높이 계산
    heights = centered @ rough_axis
    h_min, h_max = heights.min(), heights.max()
    
    # 여러 슬라이스로 나눠서 중심점 계산
    slice_heights = np.linspace(h_min + 0.05*(h_max-h_min), 
                                 h_max - 0.05*(h_max-h_min), 
                                 n_slices)
    
    centerline_points = []
    slice_info = []
    thickness = (h_max - h_min) / n_slices * 0.8
    
    for h in slice_heights:
        mask = np.abs(heights - h) < thickness / 2
        if mask.sum() > 10:
            slice_pts = centered[mask]
            slice_center = slice_pts.mean(axis=0)
            centerline_points.append(slice_center)
            slice_info.append({
                'height': h,
                'center': slice_center,
                'n_points': mask.sum()
            })
    
    if len(centerline_points) < 3:
        return None, rough_axis, []
    
    centerline_points = np.array(centerline_points)
    
    # 중심선 방향: 선형 회귀로 피팅
    # 각 좌표를 높이의 함수로 피팅
    heights_cl = np.array([s['height'] for s in slice_info])
    
    # SVD로 최적 방향 찾기
    cl_centered = centerline_points - centerline_points.mean(axis=0)
    _, _, Vt = np.linalg.svd(cl_centered)
    centerline_direction = Vt[0]  # 첫 번째 주성분 = 중심선 방향
    
    if centerline_direction @ up < 0:
        centerline_direction = -centerline_direction
    
    return centerline_points + centroid, centerline_direction, slice_info


def align_with_centerline(points, up=np.array([0, 1, 0])):
    """
    중심선 기반 정렬
    """
    centerline_pts, axis, slice_info = extract_centerline(points, n_slices=20, up=up)
    
    if centerline_pts is None:
        # 실패시 PCA로 폴백
        return align_with_pca(points, up)[:2]
    
    centroid = points.mean(axis=0)
    centered = points - centroid
    
    # 중심선 방향을 up으로 회전
    rotation_matrix = rotation_matrix_from_vectors(axis, up)
    aligned = centered @ rotation_matrix.T
    
    return aligned, {
        'method': 'centerline',
        'rotation': rotation_matrix,
        'centroid': centroid,
        'trunk_axis': axis,
        'centerline_points': centerline_pts
    }


# ============================================================================
# 방법 2: 원형성(Circularity) 최적화
# ============================================================================

def calculate_circularity(points_2d):
    """
    2D 점들의 원형성 계산 (0~1, 1이 완벽한 원)
    """
    if len(points_2d) < 5:
        return 0
    
    # 원 피팅
    model = CircleModel()
    if not model.estimate(points_2d):
        return 0
    
    cx, cy, r = model.params
    
    # 잔차 계산
    distances = np.sqrt((points_2d[:, 0] - cx)**2 + (points_2d[:, 1] - cy)**2)
    residuals = np.abs(distances - r)
    
    # 원형성: 1 - (평균 잔차 / 반지름)
    circularity = 1 - (residuals.mean() / r) if r > 0 else 0
    return max(0, min(1, circularity))


def get_slice_at_angles(points, centroid, base_axis, up, pitch, roll, height_ratio=0.5, thickness=0.02):
    """
    주어진 pitch, roll 각도로 회전된 축에서 슬라이스 추출
    """
    # pitch (x축 회전), roll (z축 회전)
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(pitch), -np.sin(pitch)],
        [0, np.sin(pitch), np.cos(pitch)]
    ])
    Rz = np.array([
        [np.cos(roll), -np.sin(roll), 0],
        [np.sin(roll), np.cos(roll), 0],
        [0, 0, 1]
    ])
    
    # 회전된 축
    rotated_axis = Rz @ Rx @ base_axis
    rotated_axis = rotated_axis / np.linalg.norm(rotated_axis)
    
    # 단면 좌표계
    tmp = up if abs(rotated_axis @ up) < 0.95 else np.array([1, 0, 0])
    u = np.cross(rotated_axis, tmp)
    u = u / np.linalg.norm(u)
    v = np.cross(rotated_axis, u)
    
    # 높이 계산
    centered = points - centroid
    heights = centered @ rotated_axis
    h_min, h_max = heights.min(), heights.max()
    target_h = h_min + (h_max - h_min) * height_ratio
    
    # 슬라이스
    mask = np.abs(heights - target_h) < thickness / 2
    if mask.sum() < 10:
        return None, rotated_axis, u, v
    
    slice_pts = centered[mask]
    points_2d = np.stack([slice_pts @ u, slice_pts @ v], axis=1)
    
    return points_2d, rotated_axis, u, v


def optimize_circularity(points, up=np.array([0, 1, 0]), height_ratio=0.5, thickness=0.02):
    """
    원형성을 최대화하는 축 각도 찾기
    """
    centroid = points.mean(axis=0)
    
    # 초기 축 (PCA)
    centered = points - centroid
    cov = np.cov(centered.T)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    idx = np.argsort(eigenvalues)[::-1]
    base_axis = eigenvectors[:, idx[0]]
    if base_axis @ up < 0:
        base_axis = -base_axis
    
    def neg_circularity(angles):
        pitch, roll = angles
        pts_2d, _, _, _ = get_slice_at_angles(
            points, centroid, base_axis, up, pitch, roll, height_ratio, thickness
        )
        if pts_2d is None or len(pts_2d) < 10:
            return 1.0  # 패널티
        
        # DBSCAN으로 노이즈 제거
        pts_clean, _ = filter_noise_dbscan(pts_2d, eps=0.05, min_samples=5)
        if len(pts_clean) < 10:
            return 1.0
        
        return -calculate_circularity(pts_clean)
    
    # 최적화 (작은 범위에서)
    from scipy.optimize import minimize
    
    result = minimize(
        neg_circularity,
        x0=[0, 0],
        method='Powell',
        bounds=[(-0.3, 0.3), (-0.3, 0.3)],  # ±17도
        options={'maxiter': 50}
    )
    
    best_pitch, best_roll = result.x
    best_circularity = -result.fun
    
    return best_pitch, best_roll, best_circularity, base_axis


def align_with_circularity(points, up=np.array([0, 1, 0]), height_ratio=0.5, thickness=0.02):
    """
    원형성 최적화 기반 정렬
    """
    centroid = points.mean(axis=0)
    centered = points - centroid
    
    # 최적 각도 찾기
    pitch, roll, circularity, base_axis = optimize_circularity(
        points, up, height_ratio, thickness
    )
    
    # 회전 행렬 계산
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(pitch), -np.sin(pitch)],
        [0, np.sin(pitch), np.cos(pitch)]
    ])
    Rz = np.array([
        [np.cos(roll), -np.sin(roll), 0],
        [np.sin(roll), np.cos(roll), 0],
        [0, 0, 1]
    ])
    
    optimized_axis = Rz @ Rx @ base_axis
    optimized_axis = optimized_axis / np.linalg.norm(optimized_axis)
    
    # 최적 축을 up으로 회전
    rotation_matrix = rotation_matrix_from_vectors(optimized_axis, up)
    aligned = centered @ rotation_matrix.T
    
    return aligned, {
        'method': 'circularity',
        'rotation': rotation_matrix,
        'centroid': centroid,
        'trunk_axis': optimized_axis,
        'pitch': pitch,
        'roll': roll,
        'circularity': circularity
    }


# ============================================================================
# 방법 3: 중심선 + 원형성 결합
# ============================================================================

def align_with_centerline_and_circularity(points, up=np.array([0, 1, 0]), height_ratio=0.5, thickness=0.02):
    """
    1단계: 중심선으로 대략적 정렬
    2단계: 원형성 최적화로 미세 조정
    """
    # 1단계: 중심선 정렬
    aligned_cl, info_cl = align_with_centerline(points, up)
    
    # 2단계: 원형성 최적화
    aligned_final, info_circ = align_with_circularity(
        aligned_cl, up, height_ratio, thickness
    )
    
    return aligned_final, {
        'method': 'centerline_circularity',
        'centerline_info': info_cl,
        'circularity_info': info_circ,
        'centroid': info_cl.get('centroid', points.mean(axis=0))
    }


# ============================================================================
# 방법 4: IMU+PCA + Centerline 결합
# ============================================================================

def align_with_imu_pca_centerline(points, imu, up=np.array([0, 1, 0])):
    """
    1단계: IMU+PCA로 대략적 정렬
    2단계: Centerline으로 미세 조정
    """
    # 1단계: IMU+PCA 정렬
    aligned_imu_pca, info_imu_pca = combined_alignment(points, imu, 'imu_then_pca')
    
    # 2단계: Centerline 정렬
    aligned_final, info_cl = align_with_centerline(aligned_imu_pca, up)
    
    return aligned_final, {
        'method': 'imu_pca_centerline',
        'imu_pca_info': info_imu_pca,
        'centerline_info': info_cl,
        'centroid': info_imu_pca.get('centroid', points.mean(axis=0))
    }


# ============================================================================
# 방법 5: IMU+PCA + Centerline + Circularity 결합 (풀 파이프라인)
# ============================================================================

def align_full_pipeline(points, imu, up=np.array([0, 1, 0]), height_ratio=0.5, thickness=0.02):
    """
    1단계: IMU+PCA로 대략적 정렬
    2단계: Centerline으로 축 보정
    3단계: Circularity로 미세 조정
    """
    # 1단계: IMU+PCA 정렬
    aligned_imu_pca, info_imu_pca = combined_alignment(points, imu, 'imu_then_pca')
    
    # 2단계: Centerline 정렬
    aligned_cl, info_cl = align_with_centerline(aligned_imu_pca, up)
    
    # 3단계: Circularity 최적화
    aligned_final, info_circ = align_with_circularity(aligned_cl, up, height_ratio, thickness)
    
    return aligned_final, {
        'method': 'full_pipeline',
        'imu_pca_info': info_imu_pca,
        'centerline_info': info_cl,
        'circularity_info': info_circ,
        'centroid': info_imu_pca.get('centroid', points.mean(axis=0))
    }


def visualize_pca_with_plane(points, pca_info, slice_3d=None, max_points=5000, title=""):
    """
    점군 + PCA 고유벡터 + 단면 평면 시각화 (새 코드에서 가져옴)
    """
    axis = pca_info['axis']
    mu = pca_info['mu']
    u = pca_info['u']
    v = pca_info['v']
    
    # 샘플링
    if len(points) > max_points:
        sample_idx = np.random.choice(len(points), max_points, replace=False)
        pts = points[sample_idx]
    else:
        pts = points
    
    fig = go.Figure()
    
    # 점군
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 2], z=pts[:, 1],
        mode='markers',
        marker=dict(size=2, color='gray', opacity=0.5),
        name='Points'
    ))
    
    # 슬라이스 포인트 (있으면)
    if slice_3d is not None and len(slice_3d) > 0:
        fig.add_trace(go.Scatter3d(
            x=slice_3d[:, 0], y=slice_3d[:, 2], z=slice_3d[:, 1],
            mode='markers',
            marker=dict(size=4, color='red'),
            name='Slice Points'
        ))
    
    # 중심점
    fig.add_trace(go.Scatter3d(
        x=[mu[0]], y=[mu[2]], z=[mu[1]],
        mode='markers',
        marker=dict(size=8, color='black'),
        name='Center (μ)'
    ))
    
    # 주축 (나무 방향)
    axis_len = 0.3
    fig.add_trace(go.Scatter3d(
        x=[mu[0] - axis[0]*axis_len, mu[0] + axis[0]*axis_len],
        y=[mu[2] - axis[2]*axis_len, mu[2] + axis[2]*axis_len],
        z=[mu[1] - axis[1]*axis_len, mu[1] + axis[1]*axis_len],
        mode='lines',
        line=dict(color='yellow', width=8),
        name='Trunk Axis'
    ))
    
    # u, v 축
    basis_len = 0.15
    fig.add_trace(go.Scatter3d(
        x=[mu[0], mu[0] + u[0]*basis_len],
        y=[mu[2], mu[2] + u[2]*basis_len],
        z=[mu[1], mu[1] + u[1]*basis_len],
        mode='lines',
        line=dict(color='red', width=6),
        name='U axis'
    ))
    fig.add_trace(go.Scatter3d(
        x=[mu[0], mu[0] + v[0]*basis_len],
        y=[mu[2], mu[2] + v[2]*basis_len],
        z=[mu[1], mu[1] + v[1]*basis_len],
        mode='lines',
        line=dict(color='blue', width=6),
        name='V axis'
    ))
    
    # 단면 평면 (사각형)
    plane_size = 0.2
    corners_2d = np.array([
        [-plane_size, -plane_size],
        [plane_size, -plane_size],
        [plane_size, plane_size],
        [-plane_size, plane_size],
        [-plane_size, -plane_size]
    ])
    corners_3d = mu + corners_2d[:, 0:1] * u + corners_2d[:, 1:2] * v
    
    fig.add_trace(go.Scatter3d(
        x=corners_3d[:, 0], y=corners_3d[:, 2], z=corners_3d[:, 1],
        mode='lines',
        line=dict(color='cyan', width=4),
        name='Cross-section Plane'
    ))
    
    # 평면 채우기
    grid_n = 10
    grid_1d = np.linspace(-plane_size, plane_size, grid_n)
    uu, vv = np.meshgrid(grid_1d, grid_1d)
    plane_pts = mu + uu.reshape(-1, 1) * u + vv.reshape(-1, 1) * v
    plane_pts = plane_pts.reshape(grid_n, grid_n, 3)
    
    fig.add_trace(go.Surface(
        x=plane_pts[:, :, 0],
        y=plane_pts[:, :, 2],
        z=plane_pts[:, :, 1],
        colorscale=[[0, 'cyan'], [1, 'cyan']],
        opacity=0.3,
        showscale=False,
        name='Plane'
    ))
    
    fig.update_layout(
        title=title or 'Point Cloud + Trunk Axis + Cross-section Plane',
        width=1000, height=800,
        scene=dict(aspectmode='data')
    )
    
    return fig


def visualize_projected_2d(points_2d, fits=None, actual_dbh=None, title=""):
    """
    투영된 2D 점들 시각화 + 피팅 결과
    """
    fig = go.Figure()
    
    # 투영된 점들
    fig.add_trace(go.Scatter(
        x=points_2d[:, 0],
        y=points_2d[:, 1],
        mode='markers',
        marker=dict(size=4, color='blue', opacity=0.6),
        name='Projected Points'
    ))
    
    # 피팅 결과 (있으면)
    if fits:
        colors = {
            'circle': '#3498db',
            'ransac_circle': '#e74c3c',
            'ellipse': '#2ecc71',
            'ransac_ellipse': '#9b59b6',
            'augmented_ellipse': '#f39c12'
        }
        
        theta = np.linspace(0, 2*np.pi, 200)
        
        for method_key, fit in fits.items():
            if fit is None:
                continue
            
            cx, cy = fit['center']
            color = colors.get(method_key, 'black')
            error = fit['dbh'] - actual_dbh if actual_dbh else 0
            
            if 'ellipse' in method_key.lower():
                a, b = fit['a'], fit['b']
                rotation = fit['theta']
                x_e = a * np.cos(theta)
                y_e = b * np.sin(theta)
                x_rot = x_e * np.cos(rotation) - y_e * np.sin(rotation) + cx
                y_rot = x_e * np.sin(rotation) + y_e * np.cos(rotation) + cy
                
                fig.add_trace(go.Scatter(
                    x=x_rot, y=y_rot,
                    mode='lines',
                    line=dict(color=color, width=2),
                    name=f'{fit["method"]}: {fit["dbh"]:.1f}cm ({error:+.1f})'
                ))
            else:
                r = fit['radius']
                fig.add_trace(go.Scatter(
                    x=cx + r * np.cos(theta),
                    y=cy + r * np.sin(theta),
                    mode='lines',
                    line=dict(color=color, width=2),
                    name=f'{fit["method"]}: {fit["dbh"]:.1f}cm ({error:+.1f})'
                ))
        
        # 실제 DBH 원 (있으면)
        if actual_dbh:
            valid_fits = [f for f in fits.values() if f is not None]
            if valid_fits:
                ref_cx = np.mean([f['center'][0] for f in valid_fits])
                ref_cy = np.mean([f['center'][1] for f in valid_fits])
                actual_r = actual_dbh / 100 / 2
                
                fig.add_trace(go.Scatter(
                    x=ref_cx + actual_r * np.cos(theta),
                    y=ref_cy + actual_r * np.sin(theta),
                    mode='lines',
                    line=dict(color='black', width=2, dash='dash'),
                    name=f'Actual: {actual_dbh:.1f}cm'
                ))
    
    fig.update_layout(
        title=title or f'Cross-section Projection ({len(points_2d)} points)',
        xaxis=dict(scaleanchor='y', scaleratio=1, title='U (m)'),
        yaxis=dict(title='V (m)'),
        width=700, height=700
    )
    
    return fig


# 피팅 함수들
def fit_circle(xz):
    if len(xz) < 3:
        return None
    model = CircleModel()
    if model.estimate(xz):
        cx, cy, r = model.params
        return {'method': 'Circle', 'center': (cx, cy), 'radius': r, 'dbh': 2*r*100, 'params': model.params}
    return None

def fit_ransac_circle(xz, min_samples=10, residual_threshold=0.015):
    if len(xz) < min_samples:
        return None
    try:
        model, inliers = ransac(xz, CircleModel, min_samples=min_samples,
                                residual_threshold=residual_threshold, max_trials=1000)
        if model:
            cx, cy, r = model.params
            return {'method': 'RANSAC Circle', 'center': (cx, cy), 'radius': r, 
                    'dbh': 2*r*100, 'params': model.params, 'inliers': inliers}
    except:
        pass
    return None

def fit_ellipse(xz):
    if len(xz) < 5:
        return None
    model = EllipseModel()
    if model.estimate(xz):
        cx, cy, a, b, theta = model.params
        return {'method': 'Ellipse', 'center': (cx, cy), 'a': a, 'b': b, 
                'theta': theta, 'dbh': (a+b)*100, 'params': model.params}
    return None

def fit_ransac_ellipse(xz, min_samples=10, residual_threshold=0.015):
    if len(xz) < min_samples:
        return None
    try:
        model, inliers = ransac(xz, EllipseModel, min_samples=min_samples,
                                residual_threshold=residual_threshold, max_trials=1000)
        if model:
            cx, cy, a, b, theta = model.params
            return {'method': 'RANSAC Ellipse', 'center': (cx, cy), 'a': a, 'b': b,
                    'theta': theta, 'dbh': (a+b)*100, 'params': model.params, 'inliers': inliers}
    except:
        pass
    return None


# ============================================================================
# 2번 코드에서 가져온 피팅 알고리즘 (Symmetric Augmentation)
# ============================================================================

def fit_circle_leastsq(points_2d):
    """최소자승법 기반 원 피팅"""
    def calc_residuals(params, points):
        cx, cy, r = params
        distances = np.sqrt((points[:, 0] - cx)**2 + (points[:, 1] - cy)**2)
        return distances - r
    
    x0 = np.median(points_2d[:, 0])
    y0 = np.median(points_2d[:, 1])
    r0 = np.std(np.linalg.norm(points_2d - [x0, y0], axis=1))
    
    result = optimize.least_squares(calc_residuals, [x0, y0, r0], args=(points_2d,))
    cx, cy, r = result.x
    return cx, cy, abs(r)


def symmetric_augment(points_2d, cx, cy):
    """대칭 증강: 점들을 중심에 대해 반사하여 완전한 원/타원 형성"""
    relative = points_2d - np.array([cx, cy])
    mirrored = -relative + np.array([cx, cy])
    return np.vstack([points_2d, mirrored])


def fit_ellipse_augmented(points_2d):
    """증강된 점들로 타원 피팅"""
    model = EllipseModel()
    success = model.estimate(points_2d)
    
    if not success:
        return None
    
    xc, yc, a, b, theta = model.params
    major = max(a, b)
    minor = min(a, b)
    
    h = ((major - minor) / (major + minor)) ** 2
    circumference = np.pi * (major + minor) * (1 + 3*h / (10 + np.sqrt(4 - 3*h)))
    
    return {
        'method': 'Symmetric Augmented Ellipse',
        'center': (xc, yc),
        'major': major * 2 * 100,
        'minor': minor * 2 * 100,
        'dbh': (major + minor) * 100,
        'circumference': circumference * 100,
        'theta': theta,
        'a': a,
        'b': b
    }


def fit_with_augmentation(points_2d):
    """대칭 증강 후 타원 피팅 (2번 코드의 핵심 알고리즘)"""
    if len(points_2d) < 3:
        return None, None
    
    cx, cy, r = fit_circle_leastsq(points_2d)
    augmented = symmetric_augment(points_2d, cx, cy)
    result = fit_ellipse_augmented(augmented)
    return result, augmented


# ============================================================================
# Plotly 시각화
# ============================================================================

def visualize_alignment_comparison(points_raw, points_imu, points_pca, points_combined, max_points=5000):
    """3가지 정렬 방법 비교 시각화"""
    
    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}],
               [{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
        subplot_titles=['Raw (원본)', 'IMU Only', 'PCA Only', 'IMU + PCA']
    )
    
    datasets = [
        (points_raw, 'Raw'),
        (points_imu, 'IMU'),
        (points_pca, 'PCA'),
        (points_combined, 'IMU+PCA')
    ]
    
    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
    
    for (pts, name), (row, col) in zip(datasets, positions):
        if len(pts) > max_points:
            idx = np.random.choice(len(pts), max_points, replace=False)
            pts_sample = pts[idx]
        else:
            pts_sample = pts
        
        # 점군
        fig.add_trace(
            go.Scatter3d(
                x=pts_sample[:, 0], 
                y=pts_sample[:, 2],  # Z → Y (화면)
                z=pts_sample[:, 1],  # Y → Z (높이)
                mode='markers',
                marker=dict(size=2, color=pts_sample[:, 1], colorscale='Viridis', opacity=0.6),
                name=name
            ),
            row=row, col=col
        )
        
        # 축 표시 (Y축 = 빨간색 = 나무 방향)
        y_range = pts[:, 1].max() - pts[:, 1].min()
        center = pts.mean(axis=0)
        
        # Y축 (높이/나무 방향) - 빨간색
        fig.add_trace(
            go.Scatter3d(
                x=[center[0], center[0]],
                y=[center[2], center[2]],
                z=[center[1] - y_range*0.3, center[1] + y_range*0.3],
                mode='lines',
                line=dict(color='red', width=8),
                name='Y-axis (Up)',
                showlegend=(row==1 and col==1)
            ),
            row=row, col=col
        )
    
    fig.update_layout(
        title='Alignment Method Comparison (Red line = Y-axis/Up direction)',
        height=900,
        width=1200
    )
    
    return fig


def visualize_aligned_tree(points, align_info, slice_3d=None, title="Aligned Tree"):
    """정렬된 나무 3D 시각화"""
    
    fig = go.Figure()
    
    # 샘플링
    max_pts = 8000
    if len(points) > max_pts:
        idx = np.random.choice(len(points), max_pts, replace=False)
        pts = points[idx]
    else:
        pts = points
    
    # 점군 (색상 = 높이)
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0],
        y=pts[:, 2],  # Z → Y
        z=pts[:, 1],  # Y → Z (높이)
        mode='markers',
        marker=dict(
            size=2,
            color=pts[:, 1],
            colorscale='Greens',
            opacity=0.6,
            colorbar=dict(title='Height (m)')
        ),
        name='Tree Points'
    ))
    
    # 슬라이스 포인트 (빨간색)
    if slice_3d is not None and len(slice_3d) > 0:
        fig.add_trace(go.Scatter3d(
            x=slice_3d[:, 0],
            y=slice_3d[:, 2],
            z=slice_3d[:, 1],
            mode='markers',
            marker=dict(size=5, color='red'),
            name='Slice Points'
        ))
    
    # 좌표축 표시
    center = points.mean(axis=0)
    axis_len = 0.2
    
    # X축 (빨간)
    fig.add_trace(go.Scatter3d(
        x=[center[0], center[0] + axis_len],
        y=[center[2], center[2]],
        z=[center[1], center[1]],
        mode='lines+text',
        line=dict(color='red', width=6),
        text=['', 'X'],
        textposition='top center',
        name='X-axis'
    ))
    
    # Y축 (초록) - 높이 방향
    fig.add_trace(go.Scatter3d(
        x=[center[0], center[0]],
        y=[center[2], center[2]],
        z=[center[1], center[1] + axis_len],
        mode='lines+text',
        line=dict(color='green', width=6),
        text=['', 'Y (Up)'],
        textposition='top center',
        name='Y-axis (Up)'
    ))
    
    # Z축 (파란)
    fig.add_trace(go.Scatter3d(
        x=[center[0], center[0]],
        y=[center[2], center[2] + axis_len],
        z=[center[1], center[1]],
        mode='lines+text',
        line=dict(color='blue', width=6),
        text=['', 'Z'],
        textposition='top center',
        name='Z-axis'
    ))
    
    # 수평면 표시 (슬라이스 높이)
    if slice_3d is not None and len(slice_3d) > 0:
        slice_height = slice_3d[:, 1].mean()
        plane_size = 0.15
        
        xx = np.linspace(center[0] - plane_size, center[0] + plane_size, 10)
        zz = np.linspace(center[2] - plane_size, center[2] + plane_size, 10)
        xx, zz = np.meshgrid(xx, zz)
        yy = np.full_like(xx, slice_height)
        
        fig.add_trace(go.Surface(
            x=xx, y=zz, z=yy,
            colorscale=[[0, 'cyan'], [1, 'cyan']],
            opacity=0.3,
            showscale=False,
            name='Slice Plane'
        ))
    
    fig.update_layout(
        title=title,
        width=1000, height=800,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Z (m)',
            zaxis_title='Y / Height (m)',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1))
        )
    )
    
    return fig


def visualize_horizontal_slice(xz_raw, xz_clean, fits, actual_dbh, title=""):
    """수평 단면 (XZ 평면) + 피팅 결과"""
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Raw vs Clean', 'Fitting Results'],
        horizontal_spacing=0.1
    )
    
    # 왼쪽: Raw vs Clean
    fig.add_trace(
        go.Scatter(
            x=xz_raw[:, 0], y=xz_raw[:, 1],
            mode='markers',
            marker=dict(size=4, color='lightgray'),
            name='Raw'
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=xz_clean[:, 0], y=xz_clean[:, 1],
            mode='markers',
            marker=dict(size=5, color='blue'),
            name='Clean (DBSCAN)'
        ),
        row=1, col=1
    )
    
    # 오른쪽: 피팅 결과
    fig.add_trace(
        go.Scatter(
            x=xz_clean[:, 0], y=xz_clean[:, 1],
            mode='markers',
            marker=dict(size=4, color='gray', opacity=0.5),
            name='Arc Data',
            showlegend=False
        ),
        row=1, col=2
    )
    
    colors = {
        'circle': '#3498db',
        'ransac_circle': '#e74c3c',
        'ellipse': '#2ecc71',
        'ransac_ellipse': '#9b59b6',
        'augmented_ellipse': '#f39c12'
    }
    
    theta = np.linspace(0, 2*np.pi, 200)
    
    for method_key, fit in fits.items():
        if fit is None:
            continue
        
        cx, cy = fit['center']
        color = colors.get(method_key, 'black')
        error = fit['dbh'] - actual_dbh
        
        if 'ellipse' in method_key.lower():
            a, b = fit['a'], fit['b']
            rotation = fit['theta']
            x_e = a * np.cos(theta)
            y_e = b * np.sin(theta)
            x_rot = x_e * np.cos(rotation) - y_e * np.sin(rotation) + cx
            y_rot = x_e * np.sin(rotation) + y_e * np.cos(rotation) + cy
            
            fig.add_trace(
                go.Scatter(
                    x=x_rot, y=y_rot,
                    mode='lines',
                    line=dict(color=color, width=2),
                    name=f'{fit["method"]}: {fit["dbh"]:.1f}cm ({error:+.1f})'
                ),
                row=1, col=2
            )
        else:
            r = fit['radius']
            fig.add_trace(
                go.Scatter(
                    x=cx + r * np.cos(theta),
                    y=cy + r * np.sin(theta),
                    mode='lines',
                    line=dict(color=color, width=2),
                    name=f'{fit["method"]}: {fit["dbh"]:.1f}cm ({error:+.1f})'
                ),
                row=1, col=2
            )
    
    # 실제 DBH 원
    if fits:
        valid_fits = [f for f in fits.values() if f is not None]
        if valid_fits:
            ref_cx = np.mean([f['center'][0] for f in valid_fits])
            ref_cy = np.mean([f['center'][1] for f in valid_fits])
            actual_r = actual_dbh / 100 / 2
            
            fig.add_trace(
                go.Scatter(
                    x=ref_cx + actual_r * np.cos(theta),
                    y=ref_cy + actual_r * np.sin(theta),
                    mode='lines',
                    line=dict(color='black', width=2, dash='dash'),
                    name=f'Actual: {actual_dbh:.1f}cm'
                ),
                row=1, col=2
            )
    
    fig.update_xaxes(scaleanchor="y", scaleratio=1, title='X (m)', row=1, col=1)
    fig.update_yaxes(title='Z (m)', row=1, col=1)
    fig.update_xaxes(scaleanchor="y", scaleratio=1, title='X (m)', row=1, col=2)
    fig.update_yaxes(title='Z (m)', row=1, col=2)
    
    fig.update_layout(
        title=title or f'Horizontal Cross-section | Actual DBH: {actual_dbh:.1f}cm',
        height=600,
        width=1200
    )
    
    return fig


def visualize_predicted_vs_true(df, method_col='pred_augmented_ellipse', save_path=None):
    """
    Predicted vs True DBH 그래프 (2번 코드에서 가져옴)
    
    Parameters:
    -----------
    df : DataFrame
        결과 데이터프레임 (actual_dbh, pred_* 컬럼 포함)
    method_col : str
        예측값 컬럼명
    save_path : str, optional
        저장 경로
    """
    # 컬럼 확인
    if method_col not in df.columns:
        print(f"Warning: {method_col} not found. Available columns: {df.columns.tolist()}")
        return None
    
    # NaN 제거
    plot_df = df[['actual_dbh', method_col]].dropna()
    
    if len(plot_df) == 0:
        print("No valid data for plotting")
        return None
    
    true_dbh = plot_df['actual_dbh'].values
    pred_dbh = plot_df[method_col].values
    
    # 선형 회귀
    slope, intercept, r_value, p_value, std_err = stats.linregress(true_dbh, pred_dbh)
    r_squared = r_value ** 2
    
    # 오차 통계
    errors = pred_dbh - true_dbh
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors**2))
    bias = np.mean(errors)
    
    print(f"\n{'='*60}")
    print(f"📊 Predicted vs True DBH Statistics ({method_col})")
    print(f"{'='*60}")
    print(f"선형 회귀: d = {slope:.3f} * d_true + {intercept:.3f}")
    print(f"결정계수 R² = {r_squared:.4f}")
    print(f"MAE: {mae:.2f} cm, RMSE: {rmse:.2f} cm, Bias: {bias:.2f} cm")
    
    # 시각화
    fig = go.Figure()
    
    x_min = true_dbh.min() - 2
    x_max = true_dbh.max() + 2
    x_range = np.linspace(x_min, x_max, 100)
    
    # ±RMSE 범위
    fig.add_trace(go.Scatter(
        x=x_range, y=x_range + rmse,
        mode='lines',
        line=dict(color='cyan', width=1, dash='dash'),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=x_range, y=x_range - rmse,
        mode='lines',
        line=dict(color='cyan', width=1, dash='dash'),
        fill='tonexty',
        fillcolor='rgba(0,255,255,0.2)',
        name=f'±RMSE ({rmse:.2f}cm)'
    ))
    
    # y=x 기준선
    fig.add_trace(go.Scatter(
        x=x_range, y=x_range,
        mode='lines',
        line=dict(color='red', width=2),
        name='y = x'
    ))
    
    # 회귀선
    fig.add_trace(go.Scatter(
        x=x_range, y=slope * x_range + intercept,
        mode='lines',
        line=dict(color='green', width=2, dash='dot'),
        name=f'Regression (R²={r_squared:.4f})'
    ))
    
    # 데이터 포인트
    fig.add_trace(go.Scatter(
        x=true_dbh,
        y=pred_dbh,
        mode='markers',
        marker=dict(size=10, color='blue', opacity=0.8),
        name='Measured',
        text=[f'Frame {i}' for i in plot_df.index],
        hovertemplate='True: %{x:.1f}cm<br>Pred: %{y:.1f}cm<br>%{text}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f'Predicted vs True DBH<br>d = {slope:.3f}·d_true + {intercept:.3f}, R² = {r_squared:.4f}, MAE = {mae:.2f}cm, RMSE = {rmse:.2f}cm',
        xaxis_title='True DBH (cm)',
        yaxis_title='Predicted DBH (cm)',
        width=800,
        height=700,
        showlegend=True
    )
    
    # 축 범위 동일하게
    fig.update_xaxes(range=[x_min, x_max])
    fig.update_yaxes(range=[x_min, x_max])
    
    if save_path:
        fig.write_html(save_path)
        print(f"💾 Saved: {save_path}")
    
    return fig


def visualize_dbh_distribution_and_comparison(df, method_col='pred_augmented_ellipse', save_path=None):
    """
    2번 코드 스타일의 (a) DBH Distribution + (b) Predicted vs True DBH 그래프
    """
    if method_col not in df.columns:
        print(f"Warning: {method_col} not found.")
        return None
    
    plot_df = df[['actual_dbh', method_col]].dropna()
    if len(plot_df) == 0:
        return None
    
    true_dbh = plot_df['actual_dbh'].values
    pred_dbh = plot_df[method_col].values
    
    # 통계
    slope, intercept, r_value, _, _ = stats.linregress(true_dbh, pred_dbh)
    r_squared = r_value ** 2
    errors = pred_dbh - true_dbh
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors**2))
    bias = np.mean(errors)
    
    # 2개 서브플롯
    fig = make_subplots(rows=1, cols=2, subplot_titles=[
        '(a) DBH Distribution',
        '(b) Predicted vs True DBH'
    ])
    
    # (a) 히스토그램
    fig.add_trace(go.Histogram(
        x=true_dbh,
        nbinsx=15,
        marker_color='steelblue',
        name='True DBH'
    ), row=1, col=1)
    
    # (b) Scatter
    x_min = true_dbh.min() - 2
    x_max = true_dbh.max() + 2
    x_range = np.linspace(x_min, x_max, 100)
    
    # ±RMSE 범위
    fig.add_trace(go.Scatter(
        x=x_range, y=x_range + rmse,
        mode='lines',
        line=dict(color='cyan', width=1, dash='dash'),
        showlegend=False
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=x_range, y=x_range - rmse,
        mode='lines',
        line=dict(color='cyan', width=1, dash='dash'),
        fill='tonexty',
        fillcolor='rgba(0,255,255,0.2)',
        name=f'±RMSE ({rmse:.2f}cm)'
    ), row=1, col=2)
    
    # y=x
    fig.add_trace(go.Scatter(
        x=x_range, y=x_range,
        mode='lines',
        line=dict(color='red', width=2),
        name='y = x'
    ), row=1, col=2)
    
    # 회귀선
    fig.add_trace(go.Scatter(
        x=x_range, y=slope * x_range + intercept,
        mode='lines',
        line=dict(color='green', width=2, dash='dot'),
        name=f'Regression (R²={r_squared:.4f})'
    ), row=1, col=2)
    
    # 데이터 포인트
    fig.add_trace(go.Scatter(
        x=true_dbh,
        y=pred_dbh,
        mode='markers',
        marker=dict(size=10, color='blue', opacity=0.8),
        name='Measured'
    ), row=1, col=2)
    
    fig.update_xaxes(title_text='d_true (cm)', row=1, col=1)
    fig.update_yaxes(title_text='Frequency', row=1, col=1)
    fig.update_xaxes(title_text='d_true (cm)', row=1, col=2)
    fig.update_yaxes(title_text='d (cm)', row=1, col=2)
    
    fig.update_layout(
        width=1000, height=450,
        title=f'DBH: d = {slope:.3f}·d_true + {intercept:.3f}, R² = {r_squared:.4f}, MAE = {mae:.2f}cm, RMSE = {rmse:.2f}cm'
    )
    
    if save_path:
        fig.write_html(save_path)
        print(f"💾 Saved: {save_path}")
    
    return fig


# ============================================================================
# 종합 분석 클래스
# ============================================================================



# ============================================================================
# 경로 설정 (필요에 따라 수정)
# ============================================================================
ROOT_PATH = "/workspace/tree"
if not os.path.exists(ROOT_PATH):
    ROOT_PATH = "."

print(f"ROOT_PATH: {ROOT_PATH}")
print("공통 설정 완료. 아래 분석 셀을 개별 실행하세요.")


### 방식 비교 (7가지 — slice / centerline / circularity / combined / imu_pca_cl / full / no_align)

In [ ]:
class VerticalTreeAnalyzer:
    """나무를 수직으로 정렬 후 분석"""
    
    def __init__(self, root_dir):
        self.root_dir = Path(root_dir)
        self.loader = TreeDataLoader(root_dir)
    
    def analyze_frame(self, frame_id, alignment_method='imu_then_pca', 
                      height_ratio=0.5, thickness=0.02,
                      extraction_method='all',
                      show_plots=False, save_dir=None, verbose=False,
                      save_dataset=False, dataset_dir=None):
        """단일 프레임 분석
        
        extraction_method:
            - 'slice': 기존 방식 (IMU+PCA 정렬 후 Y축 슬라이스)
            - 'centerline': 중심선 기반 정렬
            - 'circularity': 원형성 최적화 정렬
            - 'combined': 중심선 + 원형성 결합
            - 'imu_pca_cl': IMU+PCA + Centerline 결합 (NEW)
            - 'full': IMU+PCA + Centerline + Circularity (NEW)
            - 'no_align': 기울기 보정 없이 원본 그대로 슬라이스
            - 'all': 모든 방식 비교
        
        save_dataset: True면 슬라이스 데이터를 .bin 파일로 저장
        dataset_dir: 데이터셋 저장 경로
        """
        
        # 1. 데이터 로드
        data = self.loader.load_frame(frame_id)
        points_raw = data['points']
        imu = data['imu']
        actual_dbh = data['actual_dbh']
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"🌲 Frame {frame_id} - Analysis")
            print(f"{'='*60}")
            print(f"📊 Raw points: {len(points_raw):,}")
            print(f"📏 Actual DBH: {actual_dbh:.1f} cm")
        
        figs = {}
        fits = {}
        results_data = {}
        
        # 데이터셋 저장 경로 설정
        if save_dataset and dataset_dir:
            dataset_path = Path(dataset_dir)
            dataset_path.mkdir(parents=True, exist_ok=True)
        
        # ============================================================
        # 방법 1: 기존 슬라이스 방식 (IMU+PCA 정렬)
        # ============================================================
        if extraction_method in ['slice', 'all']:
            points_aligned, align_info = combined_alignment(points_raw, imu, 'imu_then_pca')
            xz_raw, slice_3d = extract_slice_by_ratio(points_aligned, height_ratio, thickness)
            xz_clean, _ = filter_noise_dbscan(xz_raw, eps=0.05, min_samples=5)
            
            fits['slice'] = {
                'circle': fit_circle(xz_clean),
                'ransac_circle': fit_ransac_circle(xz_clean),
                'ellipse': fit_ellipse(xz_clean),
                'ransac_ellipse': fit_ransac_ellipse(xz_clean),
                'augmented_ellipse': fit_with_augmentation(xz_clean)[0]
            }
            results_data['slice'] = {'xz_clean': xz_clean, 'slice_3d': slice_3d}
            
            # 데이터셋 저장
            if save_dataset and dataset_dir and len(xz_clean) > 0:
                save_path = dataset_path / 'slice'
                save_path.mkdir(exist_ok=True)
                xz_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
            
            if verbose:
                print(f"\n[Slice] Points: {len(xz_clean)}")
        
        # ============================================================
        # 방법 2: 중심선 기반 정렬
        # ============================================================
        if extraction_method in ['centerline', 'all']:
            aligned_cl, info_cl = align_with_centerline(points_raw)
            xz_raw_cl, slice_3d_cl = extract_slice_by_ratio(aligned_cl, height_ratio, thickness)
            xz_clean_cl, _ = filter_noise_dbscan(xz_raw_cl, eps=0.05, min_samples=5)
            
            fits['centerline'] = {
                'circle': fit_circle(xz_clean_cl),
                'ransac_circle': fit_ransac_circle(xz_clean_cl),
                'ellipse': fit_ellipse(xz_clean_cl),
                'ransac_ellipse': fit_ransac_ellipse(xz_clean_cl),
                'augmented_ellipse': fit_with_augmentation(xz_clean_cl)[0]
            }
            results_data['centerline'] = {'xz_clean': xz_clean_cl, 'slice_3d': slice_3d_cl}
            
            if save_dataset and dataset_dir and len(xz_clean_cl) > 0:
                save_path = dataset_path / 'centerline'
                save_path.mkdir(exist_ok=True)
                xz_clean_cl.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
            
            if verbose:
                print(f"[Centerline] Points: {len(xz_clean_cl)}")
        
        # ============================================================
        # 방법 3: 원형성 최적화 정렬
        # ============================================================
        if extraction_method in ['circularity', 'all']:
            aligned_circ, info_circ = align_with_circularity(points_raw, height_ratio=height_ratio, thickness=thickness)
            xz_raw_circ, slice_3d_circ = extract_slice_by_ratio(aligned_circ, height_ratio, thickness)
            xz_clean_circ, _ = filter_noise_dbscan(xz_raw_circ, eps=0.05, min_samples=5)
            
            fits['circularity'] = {
                'circle': fit_circle(xz_clean_circ),
                'ransac_circle': fit_ransac_circle(xz_clean_circ),
                'ellipse': fit_ellipse(xz_clean_circ),
                'ransac_ellipse': fit_ransac_ellipse(xz_clean_circ),
                'augmented_ellipse': fit_with_augmentation(xz_clean_circ)[0]
            }
            results_data['circularity'] = {'xz_clean': xz_clean_circ, 'slice_3d': slice_3d_circ}
            
            if save_dataset and dataset_dir and len(xz_clean_circ) > 0:
                save_path = dataset_path / 'circularity'
                save_path.mkdir(exist_ok=True)
                xz_clean_circ.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
            
            if verbose:
                circ_val = info_circ.get('circularity', 0)
                print(f"[Circularity] Points: {len(xz_clean_circ)}, Circularity: {circ_val:.3f}")
        
        # ============================================================
        # 방법 4: 중심선 + 원형성 결합
        # ============================================================
        if extraction_method in ['combined', 'all']:
            aligned_comb, info_comb = align_with_centerline_and_circularity(points_raw, height_ratio=height_ratio, thickness=thickness)
            xz_raw_comb, slice_3d_comb = extract_slice_by_ratio(aligned_comb, height_ratio, thickness)
            xz_clean_comb, _ = filter_noise_dbscan(xz_raw_comb, eps=0.05, min_samples=5)
            
            fits['combined'] = {
                'circle': fit_circle(xz_clean_comb),
                'ransac_circle': fit_ransac_circle(xz_clean_comb),
                'ellipse': fit_ellipse(xz_clean_comb),
                'ransac_ellipse': fit_ransac_ellipse(xz_clean_comb),
                'augmented_ellipse': fit_with_augmentation(xz_clean_comb)[0]
            }
            results_data['combined'] = {'xz_clean': xz_clean_comb, 'slice_3d': slice_3d_comb}
            
            if save_dataset and dataset_dir and len(xz_clean_comb) > 0:
                save_path = dataset_path / 'combined'
                save_path.mkdir(exist_ok=True)
                xz_clean_comb.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
            
            if verbose:
                print(f"[Combined] Points: {len(xz_clean_comb)}")
        
        # ============================================================
        # 방법 5: IMU+PCA + Centerline 결합 (NEW)
        # ============================================================
        if extraction_method in ['imu_pca_cl', 'all']:
            aligned_ipc, info_ipc = align_with_imu_pca_centerline(points_raw, imu)
            xz_raw_ipc, slice_3d_ipc = extract_slice_by_ratio(aligned_ipc, height_ratio, thickness)
            xz_clean_ipc, _ = filter_noise_dbscan(xz_raw_ipc, eps=0.05, min_samples=5)
            
            fits['imu_pca_cl'] = {
                'circle': fit_circle(xz_clean_ipc),
                'ransac_circle': fit_ransac_circle(xz_clean_ipc),
                'ellipse': fit_ellipse(xz_clean_ipc),
                'ransac_ellipse': fit_ransac_ellipse(xz_clean_ipc),
                'augmented_ellipse': fit_with_augmentation(xz_clean_ipc)[0]
            }
            results_data['imu_pca_cl'] = {'xz_clean': xz_clean_ipc, 'slice_3d': slice_3d_ipc}
            
            if save_dataset and dataset_dir and len(xz_clean_ipc) > 0:
                save_path = dataset_path / 'imu_pca_cl'
                save_path.mkdir(exist_ok=True)
                xz_clean_ipc.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
            
            if verbose:
                print(f"[IMU+PCA+CL] Points: {len(xz_clean_ipc)}")
        
        # ============================================================
        # 방법 6: Full Pipeline (IMU+PCA + Centerline + Circularity)
        # ============================================================
        if extraction_method in ['full', 'all']:
            aligned_full, info_full = align_full_pipeline(points_raw, imu, height_ratio=height_ratio, thickness=thickness)
            xz_raw_full, slice_3d_full = extract_slice_by_ratio(aligned_full, height_ratio, thickness)
            xz_clean_full, _ = filter_noise_dbscan(xz_raw_full, eps=0.05, min_samples=5)
            
            fits['full'] = {
                'circle': fit_circle(xz_clean_full),
                'ransac_circle': fit_ransac_circle(xz_clean_full),
                'ellipse': fit_ellipse(xz_clean_full),
                'ransac_ellipse': fit_ransac_ellipse(xz_clean_full),
                'augmented_ellipse': fit_with_augmentation(xz_clean_full)[0]
            }
            results_data['full'] = {'xz_clean': xz_clean_full, 'slice_3d': slice_3d_full}
            
            if save_dataset and dataset_dir and len(xz_clean_full) > 0:
                save_path = dataset_path / 'full'
                save_path.mkdir(exist_ok=True)
                xz_clean_full.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
            
            if verbose:
                print(f"[Full Pipeline] Points: {len(xz_clean_full)}")
        
        # ============================================================
        # 방법 7: No Alignment (보정 없음)
        # ============================================================
        if extraction_method in ['no_align', 'all']:
            noalign_2d_raw, noalign_slice_3d, noalign_info = extract_slice_no_alignment(
                points_raw, height_ratio=height_ratio, thickness=thickness
            )
            noalign_2d_clean, _ = filter_noise_dbscan(noalign_2d_raw, eps=0.05, min_samples=5)
            
            fits['no_align'] = {
                'circle': fit_circle(noalign_2d_clean),
                'ransac_circle': fit_ransac_circle(noalign_2d_clean),
                'ellipse': fit_ellipse(noalign_2d_clean),
                'ransac_ellipse': fit_ransac_ellipse(noalign_2d_clean),
                'augmented_ellipse': fit_with_augmentation(noalign_2d_clean)[0]
            }
            results_data['no_align'] = {'xz_clean': noalign_2d_clean, 'slice_3d': noalign_slice_3d}
            
            if save_dataset and dataset_dir and len(noalign_2d_clean) > 0:
                save_path = dataset_path / 'no_align'
                save_path.mkdir(exist_ok=True)
                noalign_2d_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
            
            if verbose:
                print(f"[No Align] Points: {len(noalign_2d_clean)}")
        
        return {
            'frame_id': frame_id,
            'actual_dbh': actual_dbh,
            'points_raw': points_raw,
            'fits': fits,
            'results_data': results_data,
            'figs': figs
        }
    
    def analyze_multiple_frames(self, frame_ids, alignment_method='imu_then_pca', 
                                 extraction_method='all', save_dir='vertical_analysis',
                                 verbose=False, save_dataset=False, dataset_dir=None):
        """여러 프레임 분석
        
        extraction_method: 'slice', 'centerline', 'circularity', 'combined', 
                          'imu_pca_cl', 'full', 'no_align', 'all'
        save_dataset: True면 각 방법별 슬라이스 데이터를 .bin 파일로 저장
        dataset_dir: 데이터셋 저장 경로 (예: /workspace/tree/taehee/newdataset)
        """
        
        results = []
        save_path = Path(save_dir)
        save_path.mkdir(exist_ok=True)
        
        # 데이터셋 저장 설정
        if save_dataset and dataset_dir:
            ds_path = Path(dataset_dir)
            ds_path.mkdir(parents=True, exist_ok=True)
            print(f"📁 Dataset will be saved to: {dataset_dir}")
        
        print(f"\n{'='*70}")
        print(f"🌲 Analyzing {len(frame_ids)} frames...")
        print(f"{'='*70}")
        
        for fid in tqdm(frame_ids, desc="Analyzing"):
            try:
                result = self.analyze_frame(
                    fid, 
                    alignment_method=alignment_method,
                    extraction_method=extraction_method,
                    show_plots=False, 
                    save_dir=None,
                    verbose=verbose,
                    save_dataset=save_dataset,
                    dataset_dir=dataset_dir
                )
                
                row = {
                    'frame_id': fid,
                    'actual_dbh': result['actual_dbh'],
                }
                
                # 각 방법별 결과 저장
                for ext_key, fit_dict in result['fits'].items():
                    for method, fit in fit_dict.items():
                        if fit:
                            row[f'pred_{ext_key}_{method}'] = fit['dbh']
                            row[f'error_{ext_key}_{method}'] = fit['dbh'] - result['actual_dbh']
                        else:
                            row[f'pred_{ext_key}_{method}'] = np.nan
                            row[f'error_{ext_key}_{method}'] = np.nan
                
                results.append(row)
                
            except Exception as e:
                if verbose:
                    print(f"⚠️ Frame {fid} failed: {e}")
                results.append({'frame_id': fid, 'error': str(e)})
        
        df = pd.DataFrame(results)
        df.to_csv(save_path / 'vertical_analysis_results.csv', index=False)
        
        # 라벨 파일도 데이터셋 폴더에 복사
        if save_dataset and dataset_dir:
            label_df = df[['frame_id', 'actual_dbh']].copy()
            label_df.columns = ['frame', 'dbh']
            label_df.to_csv(Path(dataset_dir) / 'labels.csv', index=False)
            print(f"💾 Labels saved to: {dataset_dir}/labels.csv")
        
        # ============================================================
        # 통계 출력
        # ============================================================
        print(f"\n{'='*70}")
        print("📊 SUMMARY STATISTICS - METHOD COMPARISON")
        print(f"{'='*70}")
        
        fitting_methods = ['augmented_ellipse']
        extraction_methods_list = []
        
        if extraction_method == 'all':
            extraction_methods_list = [
                ('slice', 'IMU+PCA'),
                ('centerline', 'Centerline'),
                ('circularity', 'Circularity'),
                ('combined', 'CL+Circ'),
                ('imu_pca_cl', 'IMU+PCA+CL'),
                ('full', 'Full Pipeline'),
                ('no_align', 'No Alignment')
            ]
        else:
            extraction_methods_list = [(extraction_method, extraction_method)]
        
        # 각 추출 방식별 통계
        summary_data = []
        for ext_key, ext_name in extraction_methods_list:
            for method in fitting_methods:
                col = f'error_{ext_key}_{method}'
                if col in df.columns:
                    valid = df[col].dropna()
                    if len(valid) > 0:
                        rmse = np.sqrt(np.mean(valid**2))
                        mae = np.mean(np.abs(valid))
                        bias = np.mean(valid)
                        summary_data.append({
                            'extraction': ext_name,
                            'fitting': method,
                            'rmse': rmse,
                            'mae': mae,
                            'bias': bias
                        })
                        print(f"  {ext_name:<20}: RMSE={rmse:.2f}cm, MAE={mae:.2f}cm, Bias={bias:+.2f}cm")
        
        # 요약 비교
        if len(summary_data) > 0:
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_csv(save_path / 'method_comparison_summary.csv', index=False)
            
            print(f"\n{'='*70}")
            print("🏆 BEST METHOD (by RMSE)")
            print(f"{'='*70}")
            
            best = summary_df.loc[summary_df['rmse'].idxmin()]
            print(f"  {best['extraction']}: RMSE={best['rmse']:.2f}cm")
        
        # ============================================================
        # 비교 그래프만 생성
        # ============================================================
        print(f"\n📈 Generating comparison plots...")
        
        self._plot_method_comparison_extended(df, extraction_methods_list, fitting_methods, save_path)
        
        return df
    
    def _plot_method_comparison_extended(self, df, extraction_methods, fitting_methods, save_path):
        """모든 추출 방식 비교 그래프"""
        
        target_fit = 'augmented_ellipse'
        n_methods = len(extraction_methods)
        
        if n_methods == 0:
            return
        
        # 2행으로 배치 (3개 이상이면)
        if n_methods <= 3:
            rows, cols = 1, n_methods
        else:
            rows, cols = 2, (n_methods + 1) // 2
        
        fig = make_subplots(
            rows=rows, cols=cols,
            subplot_titles=[name for _, name in extraction_methods]
        )
        
        colors = ['blue', 'green', 'orange', 'purple', 'red']
        
        for idx, (ext_key, ext_name) in enumerate(extraction_methods):
            pred_col = f'pred_{ext_key}_{target_fit}'
            if pred_col not in df.columns:
                continue
            
            valid_df = df[['actual_dbh', pred_col]].dropna()
            if len(valid_df) == 0:
                continue
            
            true_dbh = valid_df['actual_dbh'].values
            pred_dbh = valid_df[pred_col].values
            
            # 통계
            slope, intercept, r_value, _, _ = stats.linregress(true_dbh, pred_dbh)
            r_squared = r_value ** 2
            errors = pred_dbh - true_dbh
            rmse = np.sqrt(np.mean(errors**2))
            mae = np.mean(np.abs(errors))
            
            row = idx // cols + 1
            col = idx % cols + 1
            
            # y=x 선
            x_range = np.linspace(true_dbh.min()-2, true_dbh.max()+2, 100)
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range,
                mode='lines',
                line=dict(color='red', width=2, dash='dash'),
                name='y=x',
                showlegend=(idx==0)
            ), row=row, col=col)
            
            # ±RMSE 범위
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range + rmse,
                mode='lines',
                line=dict(color='lightgray', width=1),
                showlegend=False
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range - rmse,
                mode='lines',
                line=dict(color='lightgray', width=1),
                fill='tonexty',
                fillcolor='rgba(200,200,200,0.3)',
                showlegend=False
            ), row=row, col=col)
            
            # 데이터 포인트
            fig.add_trace(go.Scatter(
                x=true_dbh,
                y=pred_dbh,
                mode='markers',
                marker=dict(size=8, color=colors[idx % len(colors)], opacity=0.7),
                name=ext_name,
                showlegend=False,
                hovertemplate=f'True: %{{x:.1f}}cm<br>Pred: %{{y:.1f}}cm<extra>{ext_name}</extra>'
            ), row=row, col=col)
            
            # 통계 텍스트
            fig.add_trace(go.Scatter(
                x=[true_dbh.min() + 1],
                y=[pred_dbh.max() - 1],
                mode='text',
                text=[f'R²={r_squared:.3f}<br>RMSE={rmse:.2f}<br>MAE={mae:.2f}'],
                textposition='bottom right',
                textfont=dict(size=10),
                showlegend=False
            ), row=row, col=col)
            
            fig.update_xaxes(title_text='True DBH (cm)', row=row, col=col)
            fig.update_yaxes(title_text='Pred DBH (cm)', row=row, col=col)
        
        fig.update_layout(
            title=f'<b>Method Comparison - {target_fit.upper()}</b>',
            width=400*cols,
            height=450*rows,
            showlegend=True
        )
        
        fig.write_html(save_path / 'method_comparison.html')
        fig.show()
        
        # 바 차트로 RMSE 비교
        self._plot_rmse_bar_chart(df, extraction_methods, target_fit, save_path)
    
    def _plot_rmse_bar_chart(self, df, extraction_methods, target_fit, save_path):
        """RMSE 바 차트"""
        
        rmse_data = []
        mae_data = []
        names = []
        
        for ext_key, ext_name in extraction_methods:
            col = f'error_{ext_key}_{target_fit}'
            if col in df.columns:
                valid = df[col].dropna()
                if len(valid) > 0:
                    rmse_data.append(np.sqrt(np.mean(valid**2)))
                    mae_data.append(np.mean(np.abs(valid)))
                    names.append(ext_name)
        
        if not names:
            return
        
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            name='RMSE',
            x=names,
            y=rmse_data,
            marker_color='steelblue',
            text=[f'{v:.2f}' for v in rmse_data],
            textposition='outside'
        ))
        
        fig.add_trace(go.Bar(
            name='MAE',
            x=names,
            y=mae_data,
            marker_color='lightcoral',
            text=[f'{v:.2f}' for v in mae_data],
            textposition='outside'
        ))
        
        fig.update_layout(
            title='<b>Error Comparison by Method</b>',
            xaxis_title='Method',
            yaxis_title='Error (cm)',
            barmode='group',
            width=800,
            height=500
        )
        
        fig.write_html(save_path / 'error_comparison_bar.html')
        fig.show()


# ============================================================================
# 메인 실행
# ============================================================================

ROOT_PATH = "/workspace/tree"

if not os.path.exists(ROOT_PATH):
    ROOT_PATH = "."

analyzer = VerticalTreeAnalyzer(ROOT_PATH)

print("\n" + "="*70)
print("🌲 VERTICAL TREE ALIGNMENT & DBH ANALYSIS")
print("="*70)
print("\nExtraction Methods (정렬 방식):")
print("  1. slice        : 기존 IMU+PCA 정렬")
print("  2. centerline   : 중심선 기반 정렬")
print("  3. circularity  : 원형성 최적화 정렬")
print("  4. combined     : 중심선 + 원형성 결합")
print("  5. imu_pca_cl   : IMU+PCA + Centerline 결합 (NEW)")
print("  6. full         : IMU+PCA + Centerline + Circularity (NEW)")
print("  7. no_align     : 기울기 보정 없음 (기준)")
print("="*70)

# 데이터셋 저장 경로
DATASET_DIR = f"{ROOT_PATH}/taehee/newdataset"

# 여러 프레임 분석 - 7가지 방식 비교 + 데이터셋 저장
df = analyzer.analyze_multiple_frames(
    frame_ids=list(range(3, 52)),
    extraction_method='all',
    save_dir='vertical_analysis',
    verbose=False,
    save_dataset=True,  # 데이터셋 저장
    dataset_dir=DATASET_DIR  # 저장 경로
)

print(f"\n✅ Dataset saved to: {DATASET_DIR}/")
print("   각 방법별 폴더 구조:")
print("   - slice/data_XXXX.bin")
print("   - centerline/data_XXXX.bin")
print("   - circularity/data_XXXX.bin")
print("   - combined/data_XXXX.bin")
print("   - imu_pca_cl/data_XXXX.bin")
print("   - full/data_XXXX.bin")
print("   - no_align/data_XXXX.bin")
print("   - labels.csv")

### IMU + PCA & no_align 비교 (mid vs mean)

In [ ]:
class VerticalTreeAnalyzer:
    """나무를 수직으로 정렬 후 분석"""
    
    def __init__(self, root_dir):
        self.root_dir = Path(root_dir)
        self.loader = TreeDataLoader(root_dir)
    
    def analyze_frame(self, frame_id, alignment_method='imu_then_pca', 
                      height_ratio=0.5, thickness=0.02,
                      extraction_method='all',
                      show_plots=False, save_dir=None, verbose=False,
                      save_dataset=False, dataset_dir=None):
        """단일 프레임 분석
        
        extraction_method:
            - 'slice': IMU+PCA 정렬 (mid_slice)
            - 'no_align': 보정 없음 (mid_slice)
            - 'mean_slice': 30~70% 높이 평균 슬라이스
            - 'all': 모든 방식 비교 (4가지)
        """
        
        # 1. 데이터 로드
        data = self.loader.load_frame(frame_id)
        points_raw = data['points']
        imu = data['imu']
        actual_dbh = data['actual_dbh']
        
        if verbose:
            print(f"Frame {frame_id}: {len(points_raw):,} points, DBH={actual_dbh:.1f}cm")
        
        fits = {}
        results_data = {}
        
        # 데이터셋 저장 경로 설정
        if save_dataset and dataset_dir:
            dataset_path = Path(dataset_dir)
            dataset_path.mkdir(parents=True, exist_ok=True)
        
        # ============================================================
        # 방법 1: IMU+PCA 정렬 - Mid Slice (50%)
        # ============================================================
        if extraction_method in ['slice', 'all']:
            points_aligned, _ = combined_alignment(points_raw, imu, 'imu_then_pca')
            xz_raw, slice_3d = extract_slice_by_ratio(points_aligned, height_ratio, thickness)
            xz_clean, _ = filter_noise_dbscan(xz_raw, eps=0.05, min_samples=5)
            
            fits['mid_slice_imu_pca'] = {
                'circle': fit_circle(xz_clean),
                'ransac_circle': fit_ransac_circle(xz_clean),
                'ellipse': fit_ellipse(xz_clean),
                'ransac_ellipse': fit_ransac_ellipse(xz_clean),
                'augmented_ellipse': fit_with_augmentation(xz_clean)[0]
            }
            results_data['mid_slice_imu_pca'] = {'xz_clean': xz_clean, 'slice_3d': slice_3d}
            
            if save_dataset and dataset_dir and len(xz_clean) > 0:
                save_path = dataset_path / 'mid_slice_imu_pca'
                save_path.mkdir(exist_ok=True)
                xz_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
        
        # ============================================================
        # 방법 2: No Alignment - Mid Slice (50%)
        # ============================================================
        if extraction_method in ['no_align', 'all']:
            noalign_2d_raw, noalign_slice_3d, _ = extract_slice_no_alignment(
                points_raw, height_ratio=height_ratio, thickness=thickness
            )
            noalign_2d_clean, _ = filter_noise_dbscan(noalign_2d_raw, eps=0.05, min_samples=5)
            
            fits['mid_slice_no_align'] = {
                'circle': fit_circle(noalign_2d_clean),
                'ransac_circle': fit_ransac_circle(noalign_2d_clean),
                'ellipse': fit_ellipse(noalign_2d_clean),
                'ransac_ellipse': fit_ransac_ellipse(noalign_2d_clean),
                'augmented_ellipse': fit_with_augmentation(noalign_2d_clean)[0]
            }
            results_data['mid_slice_no_align'] = {'xz_clean': noalign_2d_clean, 'slice_3d': noalign_slice_3d}
            
            if save_dataset and dataset_dir and len(noalign_2d_clean) > 0:
                save_path = dataset_path / 'mid_slice_no_align'
                save_path.mkdir(exist_ok=True)
                noalign_2d_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
        
        # ============================================================
        # 방법 3: IMU+PCA - Mean Slice (30~70% 평균)
        # ============================================================
        if extraction_method in ['mean_slice', 'all']:
            points_aligned, _ = combined_alignment(points_raw, imu, 'imu_then_pca')
            mean_xz_imu = self._extract_mean_slice(points_aligned, height_range=(0.3, 0.7), n_slices=10, thickness=thickness)
            mean_xz_imu_clean, _ = filter_noise_dbscan(mean_xz_imu, eps=0.05, min_samples=5)
            
            fits['mean_slice_imu_pca'] = {
                'circle': fit_circle(mean_xz_imu_clean),
                'ransac_circle': fit_ransac_circle(mean_xz_imu_clean),
                'ellipse': fit_ellipse(mean_xz_imu_clean),
                'ransac_ellipse': fit_ransac_ellipse(mean_xz_imu_clean),
                'augmented_ellipse': fit_with_augmentation(mean_xz_imu_clean)[0]
            }
            results_data['mean_slice_imu_pca'] = {'xz_clean': mean_xz_imu_clean}
            
            if save_dataset and dataset_dir and len(mean_xz_imu_clean) > 0:
                save_path = dataset_path / 'mean_slice_imu_pca'
                save_path.mkdir(exist_ok=True)
                mean_xz_imu_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
        
        # ============================================================
        # 방법 4: No Alignment - Mean Slice (30~70% 평균)
        # ============================================================
        if extraction_method in ['mean_slice', 'all']:
            mean_xz_noalign = self._extract_mean_slice_no_align(points_raw, height_range=(0.3, 0.7), n_slices=10, thickness=thickness)
            mean_xz_noalign_clean, _ = filter_noise_dbscan(mean_xz_noalign, eps=0.05, min_samples=5)
            
            fits['mean_slice_no_align'] = {
                'circle': fit_circle(mean_xz_noalign_clean),
                'ransac_circle': fit_ransac_circle(mean_xz_noalign_clean),
                'ellipse': fit_ellipse(mean_xz_noalign_clean),
                'ransac_ellipse': fit_ransac_ellipse(mean_xz_noalign_clean),
                'augmented_ellipse': fit_with_augmentation(mean_xz_noalign_clean)[0]
            }
            results_data['mean_slice_no_align'] = {'xz_clean': mean_xz_noalign_clean}
            
            if save_dataset and dataset_dir and len(mean_xz_noalign_clean) > 0:
                save_path = dataset_path / 'mean_slice_no_align'
                save_path.mkdir(exist_ok=True)
                mean_xz_noalign_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
        
        return {
            'frame_id': frame_id,
            'actual_dbh': actual_dbh,
            'points_raw': points_raw,
            'fits': fits,
            'results_data': results_data
        }
    
    def _extract_mean_slice(self, points_aligned, height_range=(0.3, 0.7), n_slices=10, thickness=0.02):
        """
        30~70% 높이에서 여러 슬라이스를 추출하고 평균 중심으로 정렬하여 합침
        """
        y_vals = points_aligned[:, 1]
        y_min, y_max = y_vals.min(), y_vals.max()
        
        # 슬라이스 높이들
        slice_heights = np.linspace(
            y_min + (y_max - y_min) * height_range[0],
            y_min + (y_max - y_min) * height_range[1],
            n_slices
        )
        
        all_points = []
        for h in slice_heights:
            mask = np.abs(y_vals - h) < thickness / 2
            if mask.sum() > 5:
                slice_pts = points_aligned[mask][:, [0, 2]]  # XZ
                # 각 슬라이스의 중심을 원점으로
                center = slice_pts.mean(axis=0)
                slice_centered = slice_pts - center
                all_points.append(slice_centered)
        
        if len(all_points) == 0:
            return np.array([]).reshape(0, 2)
        
        return np.vstack(all_points)
    
    def _extract_mean_slice_no_align(self, points_raw, height_range=(0.3, 0.7), n_slices=10, thickness=0.02):
        """
        보정 없이 30~70% 높이에서 여러 슬라이스 추출 후 평균
        """
        centroid = points_raw.mean(axis=0)
        centered = points_raw - centroid
        
        y_vals = centered[:, 1]
        y_min, y_max = y_vals.min(), y_vals.max()
        
        slice_heights = np.linspace(
            y_min + (y_max - y_min) * height_range[0],
            y_min + (y_max - y_min) * height_range[1],
            n_slices
        )
        
        all_points = []
        for h in slice_heights:
            mask = np.abs(y_vals - h) < thickness / 2
            if mask.sum() > 5:
                slice_pts = centered[mask][:, [0, 2]]  # XZ
                center = slice_pts.mean(axis=0)
                slice_centered = slice_pts - center
                all_points.append(slice_centered)
        
        if len(all_points) == 0:
            return np.array([]).reshape(0, 2)
        
        return np.vstack(all_points)
    
    def analyze_multiple_frames(self, frame_ids, alignment_method='imu_then_pca', 
                                 extraction_method='all', save_dir='vertical_analysis',
                                 verbose=False, save_dataset=False, dataset_dir=None):
        """여러 프레임 분석
        
        4가지 방식 비교:
        1. mid_slice_imu_pca: IMU+PCA 정렬 + 50% 높이 슬라이스
        2. mid_slice_no_align: 보정 없음 + 50% 높이 슬라이스
        3. mean_slice_imu_pca: IMU+PCA 정렬 + 30~70% 평균 슬라이스
        4. mean_slice_no_align: 보정 없음 + 30~70% 평균 슬라이스
        """
        
        results = []
        save_path = Path(save_dir)
        save_path.mkdir(exist_ok=True)
        
        # 데이터셋 저장 설정
        if save_dataset and dataset_dir:
            ds_path = Path(dataset_dir)
            ds_path.mkdir(parents=True, exist_ok=True)
            print(f"📁 Dataset will be saved to: {dataset_dir}")
        
        print(f"\n{'='*70}")
        print(f"🌲 Analyzing {len(frame_ids)} frames...")
        print(f"{'='*70}")
        
        # 시각화용 샘플 저장
        sample_data = {}
        sample_frame = frame_ids[len(frame_ids)//2]  # 중간 프레임 선택
        
        for fid in tqdm(frame_ids, desc="Analyzing"):
            try:
                result = self.analyze_frame(
                    fid, 
                    alignment_method=alignment_method,
                    extraction_method=extraction_method,
                    show_plots=False, 
                    save_dir=None,
                    verbose=verbose,
                    save_dataset=save_dataset,
                    dataset_dir=dataset_dir
                )
                
                row = {
                    'frame_id': fid,
                    'actual_dbh': result['actual_dbh'],
                }
                
                # 각 방법별 결과 저장
                for ext_key, fit_dict in result['fits'].items():
                    for method, fit in fit_dict.items():
                        if fit:
                            row[f'pred_{ext_key}_{method}'] = fit['dbh']
                            row[f'error_{ext_key}_{method}'] = fit['dbh'] - result['actual_dbh']
                        else:
                            row[f'pred_{ext_key}_{method}'] = np.nan
                            row[f'error_{ext_key}_{method}'] = np.nan
                
                results.append(row)
                
                # 샘플 데이터 저장 (시각화용)
                if fid == sample_frame:
                    sample_data = {
                        'frame_id': fid,
                        'actual_dbh': result['actual_dbh'],
                        'results_data': result['results_data'],
                        'fits': result['fits']
                    }
                
            except Exception as e:
                if verbose:
                    print(f"⚠️ Frame {fid} failed: {e}")
                results.append({'frame_id': fid, 'error': str(e)})
        
        df = pd.DataFrame(results)
        df.to_csv(save_path / 'vertical_analysis_results.csv', index=False)
        
        # 라벨 파일도 데이터셋 폴더에 복사
        if save_dataset and dataset_dir:
            label_df = df[['frame_id', 'actual_dbh']].copy()
            label_df.columns = ['frame', 'dbh']
            label_df.to_csv(Path(dataset_dir) / 'labels.csv', index=False)
            print(f"💾 Labels saved to: {dataset_dir}/labels.csv")
        
        # ============================================================
        # 통계 출력
        # ============================================================
        print(f"\n{'='*70}")
        print("📊 SUMMARY STATISTICS - 4 METHODS COMPARISON")
        print(f"{'='*70}")
        
        fitting_methods = ['augmented_ellipse']
        extraction_methods_list = [
            ('mid_slice_imu_pca', 'Mid Slice + IMU+PCA'),
            ('mid_slice_no_align', 'Mid Slice + No Align'),
            ('mean_slice_imu_pca', 'Mean Slice + IMU+PCA'),
            ('mean_slice_no_align', 'Mean Slice + No Align')
        ]
        
        # 각 추출 방식별 통계
        summary_data = []
        for ext_key, ext_name in extraction_methods_list:
            for method in fitting_methods:
                col = f'error_{ext_key}_{method}'
                if col in df.columns:
                    valid = df[col].dropna()
                    if len(valid) > 0:
                        rmse = np.sqrt(np.mean(valid**2))
                        mae = np.mean(np.abs(valid))
                        bias = np.mean(valid)
                        summary_data.append({
                            'extraction': ext_name,
                            'fitting': method,
                            'rmse': rmse,
                            'mae': mae,
                            'bias': bias
                        })
                        print(f"  {ext_name:<25}: RMSE={rmse:.2f}cm, MAE={mae:.2f}cm, Bias={bias:+.2f}cm")
        
        # 요약 비교
        if len(summary_data) > 0:
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_csv(save_path / 'method_comparison_summary.csv', index=False)
            
            print(f"\n{'='*70}")
            print("🏆 BEST METHOD (by RMSE)")
            print(f"{'='*70}")
            
            best = summary_df.loc[summary_df['rmse'].idxmin()]
            print(f"  {best['extraction']}: RMSE={best['rmse']:.2f}cm")
        
        # ============================================================
        # 비교 그래프 생성
        # ============================================================
        print(f"\n📈 Generating comparison plots...")
        
        self._plot_method_comparison_4(df, extraction_methods_list, fitting_methods, save_path)
        
        # ============================================================
        # Mean Slice 시각화 (IMU+PCA vs No Align)
        # ============================================================
        if sample_data:
            print(f"\n🎨 Visualizing Mean Slice samples (Frame {sample_frame})...")
            self._visualize_mean_slice_samples(sample_data, save_path)
        
        return df
    
    def _plot_method_comparison_4(self, df, extraction_methods, fitting_methods, save_path):
        """4가지 추출 방식 비교 그래프"""
        
        target_fit = 'augmented_ellipse'
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[name for _, name in extraction_methods]
        )
        
        colors = ['blue', 'red', 'green', 'orange']
        
        for idx, (ext_key, ext_name) in enumerate(extraction_methods):
            pred_col = f'pred_{ext_key}_{target_fit}'
            if pred_col not in df.columns:
                continue
            
            valid_df = df[['actual_dbh', pred_col]].dropna()
            if len(valid_df) == 0:
                continue
            
            true_dbh = valid_df['actual_dbh'].values
            pred_dbh = valid_df[pred_col].values
            
            # 통계
            slope, intercept, r_value, _, _ = stats.linregress(true_dbh, pred_dbh)
            r_squared = r_value ** 2
            errors = pred_dbh - true_dbh
            rmse = np.sqrt(np.mean(errors**2))
            mae = np.mean(np.abs(errors))
            
            row = idx // 2 + 1
            col = idx % 2 + 1
            
            # y=x 선
            x_range = np.linspace(true_dbh.min()-2, true_dbh.max()+2, 100)
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range,
                mode='lines',
                line=dict(color='gray', width=2, dash='dash'),
                name='y=x',
                showlegend=(idx==0)
            ), row=row, col=col)
            
            # ±RMSE 범위
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range + rmse,
                mode='lines',
                line=dict(color='lightgray', width=1),
                showlegend=False
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range - rmse,
                mode='lines',
                line=dict(color='lightgray', width=1),
                fill='tonexty',
                fillcolor='rgba(200,200,200,0.3)',
                showlegend=False
            ), row=row, col=col)
            
            # 데이터 포인트
            fig.add_trace(go.Scatter(
                x=true_dbh,
                y=pred_dbh,
                mode='markers',
                marker=dict(size=8, color=colors[idx], opacity=0.7),
                name=ext_name,
                showlegend=False,
                hovertemplate=f'True: %{{x:.1f}}cm<br>Pred: %{{y:.1f}}cm<extra>{ext_name}</extra>'
            ), row=row, col=col)
            
            # 통계 텍스트
            fig.add_trace(go.Scatter(
                x=[true_dbh.min() + 1],
                y=[pred_dbh.max() - 1],
                mode='text',
                text=[f'R²={r_squared:.3f}<br>RMSE={rmse:.2f}<br>MAE={mae:.2f}'],
                textposition='bottom right',
                textfont=dict(size=10),
                showlegend=False
            ), row=row, col=col)
            
            fig.update_xaxes(title_text='True DBH (cm)', row=row, col=col)
            fig.update_yaxes(title_text='Pred DBH (cm)', row=row, col=col)
        
        fig.update_layout(
            title='<b>Method Comparison - Augmented Ellipse</b>',
            width=900,
            height=800,
            showlegend=True
        )
        
        fig.write_html(save_path / 'method_comparison.html')
        fig.show()
        
        # 바 차트
        self._plot_rmse_bar_chart_4(df, extraction_methods, target_fit, save_path)
    
    def _plot_rmse_bar_chart_4(self, df, extraction_methods, target_fit, save_path):
        """RMSE 바 차트"""
        
        rmse_data = []
        mae_data = []
        names = []
        
        for ext_key, ext_name in extraction_methods:
            col = f'error_{ext_key}_{target_fit}'
            if col in df.columns:
                valid = df[col].dropna()
                if len(valid) > 0:
                    rmse_data.append(np.sqrt(np.mean(valid**2)))
                    mae_data.append(np.mean(np.abs(valid)))
                    names.append(ext_name)
        
        if not names:
            return
        
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            name='RMSE',
            x=names,
            y=rmse_data,
            marker_color='steelblue',
            text=[f'{v:.2f}' for v in rmse_data],
            textposition='outside'
        ))
        
        fig.add_trace(go.Bar(
            name='MAE',
            x=names,
            y=mae_data,
            marker_color='lightcoral',
            text=[f'{v:.2f}' for v in mae_data],
            textposition='outside'
        ))
        
        fig.update_layout(
            title='<b>Error Comparison by Method</b>',
            xaxis_title='Method',
            yaxis_title='Error (cm)',
            barmode='group',
            width=800,
            height=500
        )
        
        fig.write_html(save_path / 'error_comparison_bar.html')
        fig.show()
    
    def _visualize_mean_slice_samples(self, sample_data, save_path):
        """Mean Slice 시각화 - IMU+PCA vs No Align"""
        
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=['Mean Slice + IMU+PCA', 'Mean Slice + No Align']
        )
        
        frame_id = sample_data['frame_id']
        actual_dbh = sample_data['actual_dbh']
        
        # IMU+PCA 버전
        if 'mean_slice_imu_pca' in sample_data['results_data']:
            xz_imu = sample_data['results_data']['mean_slice_imu_pca']['xz_clean']
            fit_imu = sample_data['fits'].get('mean_slice_imu_pca', {}).get('augmented_ellipse')
            
            fig.add_trace(go.Scatter(
                x=xz_imu[:, 0] * 100,  # m -> cm
                y=xz_imu[:, 1] * 100,
                mode='markers',
                marker=dict(size=3, color='blue', opacity=0.5),
                name='Points (IMU+PCA)'
            ), row=1, col=1)
            
            # 피팅 결과
            if fit_imu:
                cx, cy = fit_imu['center']
                a, b = fit_imu['a'], fit_imu['b']
                theta_fit = fit_imu['theta']
                t = np.linspace(0, 2*np.pi, 200)
                x_e = a * np.cos(t)
                y_e = b * np.sin(t)
                x_rot = (x_e * np.cos(theta_fit) - y_e * np.sin(theta_fit) + cx) * 100
                y_rot = (x_e * np.sin(theta_fit) + y_e * np.cos(theta_fit) + cy) * 100
                
                fig.add_trace(go.Scatter(
                    x=x_rot, y=y_rot,
                    mode='lines',
                    line=dict(color='red', width=2),
                    name=f'Fit: {fit_imu["dbh"]:.1f}cm'
                ), row=1, col=1)
        
        # No Align 버전
        if 'mean_slice_no_align' in sample_data['results_data']:
            xz_noalign = sample_data['results_data']['mean_slice_no_align']['xz_clean']
            fit_noalign = sample_data['fits'].get('mean_slice_no_align', {}).get('augmented_ellipse')
            
            fig.add_trace(go.Scatter(
                x=xz_noalign[:, 0] * 100,
                y=xz_noalign[:, 1] * 100,
                mode='markers',
                marker=dict(size=3, color='orange', opacity=0.5),
                name='Points (No Align)'
            ), row=1, col=2)
            
            if fit_noalign:
                cx, cy = fit_noalign['center']
                a, b = fit_noalign['a'], fit_noalign['b']
                theta_fit = fit_noalign['theta']
                t = np.linspace(0, 2*np.pi, 200)
                x_e = a * np.cos(t)
                y_e = b * np.sin(t)
                x_rot = (x_e * np.cos(theta_fit) - y_e * np.sin(theta_fit) + cx) * 100
                y_rot = (x_e * np.sin(theta_fit) + y_e * np.cos(theta_fit) + cy) * 100
                
                fig.add_trace(go.Scatter(
                    x=x_rot, y=y_rot,
                    mode='lines',
                    line=dict(color='red', width=2),
                    name=f'Fit: {fit_noalign["dbh"]:.1f}cm'
                ), row=1, col=2)
        
        # 실제 DBH 원 (참고용)
        actual_r = actual_dbh / 2
        theta = np.linspace(0, 2*np.pi, 100)
        
        fig.add_trace(go.Scatter(
            x=actual_r * np.cos(theta),
            y=actual_r * np.sin(theta),
            mode='lines',
            line=dict(color='black', width=2, dash='dash'),
            name=f'Actual: {actual_dbh:.1f}cm'
        ), row=1, col=1)
        
        fig.add_trace(go.Scatter(
            x=actual_r * np.cos(theta),
            y=actual_r * np.sin(theta),
            mode='lines',
            line=dict(color='black', width=2, dash='dash'),
            showlegend=False
        ), row=1, col=2)
        
        fig.update_xaxes(title_text='X (cm)', scaleanchor='y', scaleratio=1, row=1, col=1)
        fig.update_yaxes(title_text='Y (cm)', row=1, col=1)
        fig.update_xaxes(title_text='X (cm)', scaleanchor='y', scaleratio=1, row=1, col=2)
        fig.update_yaxes(title_text='Y (cm)', row=1, col=2)
        
        fig.update_layout(
            title=f'<b>Mean Slice Visualization - Frame {frame_id} (Actual DBH: {actual_dbh:.1f}cm)</b>',
            width=1000,
            height=500,
            showlegend=True
        )
        
        fig.write_html(save_path / 'mean_slice_visualization.html')
        fig.show()


# ============================================================================
# 메인 실행
# ============================================================================

ROOT_PATH = "/workspace/tree"

if not os.path.exists(ROOT_PATH):
    ROOT_PATH = "."

analyzer = VerticalTreeAnalyzer(ROOT_PATH)

print("\n" + "="*70)
print("🌲 VERTICAL TREE ALIGNMENT & DBH ANALYSIS")
print("="*70)
print("\n4가지 방식 비교:")
print("  1. mid_slice_imu_pca   : IMU+PCA 정렬 + 50% 높이 슬라이스")
print("  2. mid_slice_no_align  : 보정 없음 + 50% 높이 슬라이스")
print("  3. mean_slice_imu_pca  : IMU+PCA 정렬 + 30~70% 평균 슬라이스")
print("  4. mean_slice_no_align : 보정 없음 + 30~70% 평균 슬라이스")
print("="*70)

# 데이터셋 저장 경로
DATASET_DIR = f"{ROOT_PATH}/taehee/newdataset"

# 여러 프레임 분석 - 4가지 방식 비교 + 데이터셋 저장
df = analyzer.analyze_multiple_frames(
    frame_ids=list(range(3, 52)),
    extraction_method='all',
    save_dir='vertical_analysis',
    verbose=False,
    save_dataset=True,
    dataset_dir=DATASET_DIR
)

print(f"\n✅ Dataset saved to: {DATASET_DIR}/")
print("   폴더 구조:")
print("   - mid_slice_imu_pca/data_XXXX.bin")
print("   - mid_slice_no_align/data_XXXX.bin")
print("   - mean_slice_imu_pca/data_XXXX.bin")
print("   - mean_slice_no_align/data_XXXX.bin")
print("   - labels.csv")

### mean 범위별 오차율 비교

> ⚠️ 이 분석은 `label_filtered.csv` (루트 경로 직접)를 사용합니다.

In [ ]:
class VerticalTreeAnalyzer:
    """나무를 수직으로 정렬 후 분석"""
    
    def __init__(self, root_dir):
        self.root_dir = Path(root_dir)
        self.loader = TreeDataLoader(root_dir, label_csv='label_filtered.csv')
    
    def analyze_frame(self, frame_id, height_ratio=0.5, thickness=0.02,
                      height_ranges=None, verbose=False):
        """단일 프레임 분석 - Mean Slice 범위별 비교
        
        height_ranges: 슬라이스 범위 리스트 [(0.0, 1.0), (0.1, 0.9), ...]
        """
        
        if height_ranges is None:
            height_ranges = [
                (0.0, 1.0),   # 0~100%
                (0.1, 0.9),   # 10~90%
                (0.2, 0.8),   # 20~80%
                (0.3, 0.7),   # 30~70%
                (0.4, 0.6),   # 40~60%
            ]
        
        # 데이터 로드
        data = self.loader.load_frame(frame_id)
        points_raw = data['points']
        imu = data['imu']
        actual_dbh = data['actual_dbh']
        
        if verbose:
            print(f"Frame {frame_id}: {len(points_raw):,} points, DBH={actual_dbh:.1f}cm")
        
        # IMU+PCA 정렬
        points_aligned, _ = combined_alignment(points_raw, imu, 'imu_then_pca')
        
        fits = {}
        results_data = {}
        
        # 각 범위별 Mean Slice 추출 및 피팅
        for h_range in height_ranges:
            range_name = f'{int(h_range[0]*100)}_{int(h_range[1]*100)}'
            
            mean_xz = self._extract_mean_slice(points_aligned, height_range=h_range, n_slices=10, thickness=thickness)
            mean_xz_clean, _ = filter_noise_dbscan(mean_xz, eps=0.05, min_samples=5)
            
            if len(mean_xz_clean) > 10:
                fits[range_name] = {
                    'augmented_ellipse': fit_with_augmentation(mean_xz_clean)[0]
                }
                results_data[range_name] = {'xz_clean': mean_xz_clean}
            else:
                fits[range_name] = {'augmented_ellipse': None}
                results_data[range_name] = {'xz_clean': np.array([]).reshape(0, 2)}
        
        return {
            'frame_id': frame_id,
            'actual_dbh': actual_dbh,
            'fits': fits,
            'results_data': results_data
        }
    
    def _extract_mean_slice(self, points_aligned, height_range=(0.3, 0.7), n_slices=10, thickness=0.02):
        """
        지정된 높이 범위에서 여러 슬라이스를 추출하고 평균 중심으로 정렬하여 합침
        """
        y_vals = points_aligned[:, 1]
        y_min, y_max = y_vals.min(), y_vals.max()
        
        slice_heights = np.linspace(
            y_min + (y_max - y_min) * height_range[0],
            y_min + (y_max - y_min) * height_range[1],
            n_slices
        )
        
        all_points = []
        for h in slice_heights:
            mask = np.abs(y_vals - h) < thickness / 2
            if mask.sum() > 5:
                slice_pts = points_aligned[mask][:, [0, 2]]  # XZ
                center = slice_pts.mean(axis=0)
                slice_centered = slice_pts - center
                all_points.append(slice_centered)
        
        if len(all_points) == 0:
            return np.array([]).reshape(0, 2)
        
        return np.vstack(all_points)
    
    def analyze_multiple_frames(self, frame_ids, save_dir='vertical_analysis', verbose=False):
        """여러 프레임 분석 - Mean Slice 범위별 비교
        
        5가지 범위 비교:
        1. 0~100%
        2. 10~90%
        3. 20~80%
        4. 30~70%
        5. 40~60%
        """
        
        height_ranges = [
            (0.0, 1.0),   # 0~100%
            (0.1, 0.9),   # 10~90%
            (0.2, 0.8),   # 20~80%
            (0.3, 0.7),   # 30~70%
            (0.4, 0.6),   # 40~60%
        ]
        
        results = []
        save_path = Path(save_dir)
        save_path.mkdir(exist_ok=True)
        
        print(f"\n{'='*70}")
        print(f"🌲 Analyzing {len(frame_ids)} frames - Mean Slice Range Comparison")
        print(f"{'='*70}")
        
        for fid in tqdm(frame_ids, desc="Analyzing"):
            try:
                result = self.analyze_frame(fid, height_ranges=height_ranges, verbose=verbose)
                
                row = {
                    'frame_id': fid,
                    'actual_dbh': result['actual_dbh'],
                }
                
                for ext_key, fit_dict in result['fits'].items():
                    fit = fit_dict.get('augmented_ellipse')
                    if fit:
                        row[f'pred_{ext_key}'] = fit['dbh']
                        row[f'error_{ext_key}'] = fit['dbh'] - result['actual_dbh']
                    else:
                        row[f'pred_{ext_key}'] = np.nan
                        row[f'error_{ext_key}'] = np.nan
                
                results.append(row)
                
            except Exception as e:
                if verbose:
                    print(f"⚠️ Frame {fid} failed: {e}")
                results.append({'frame_id': fid, 'error': str(e)})
        
        df = pd.DataFrame(results)
        df.to_csv(save_path / 'range_comparison_results.csv', index=False)
        
        # ============================================================
        # 통계 출력
        # ============================================================
        print(f"\n{'='*70}")
        print("📊 SUMMARY - Mean Slice Range Comparison (IMU+PCA)")
        print(f"{'='*70}")
        
        range_names = [
            ('0_100', '0~100%'),
            ('10_90', '10~90%'),
            ('20_80', '20~80%'),
            ('30_70', '30~70%'),
            ('40_60', '40~60%'),
        ]
        
        summary_data = []
        for ext_key, ext_name in range_names:
            col = f'error_{ext_key}'
            if col in df.columns:
                valid = df[col].dropna()
                if len(valid) > 0:
                    rmse = np.sqrt(np.mean(valid**2))
                    mae = np.mean(np.abs(valid))
                    bias = np.mean(valid)
                    summary_data.append({
                        'range': ext_name,
                        'rmse': rmse,
                        'mae': mae,
                        'bias': bias
                    })
                    print(f"  {ext_name:<15}: RMSE={rmse:.2f}cm, MAE={mae:.2f}cm, Bias={bias:+.2f}cm")
        
        if len(summary_data) > 0:
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_csv(save_path / 'range_comparison_summary.csv', index=False)
            
            print(f"\n{'='*70}")
            print("🏆 BEST RANGE (by RMSE)")
            print(f"{'='*70}")
            
            best = summary_df.loc[summary_df['rmse'].idxmin()]
            print(f"  {best['range']}: RMSE={best['rmse']:.2f}cm")
        
        # ============================================================
        # 비교 그래프 생성
        # ============================================================
        print(f"\n📈 Generating comparison plots...")
        
        self._plot_range_comparison(df, range_names, save_path)
        
        return df
    
    def _plot_range_comparison(self, df, range_names, save_path):
        """5가지 범위 비교 그래프"""
        
        # 2x3 레이아웃 (5개 + 빈칸)
        fig = make_subplots(
            rows=2, cols=3,
            subplot_titles=[name for _, name in range_names] + [''],
            specs=[[{}, {}, {}], [{}, {}, {}]]
        )
        
        colors = ['blue', 'green', 'orange', 'red', 'purple']
        
        for idx, (ext_key, ext_name) in enumerate(range_names):
            pred_col = f'pred_{ext_key}'
            if pred_col not in df.columns:
                continue
            
            valid_df = df[['actual_dbh', pred_col]].dropna()
            if len(valid_df) == 0:
                continue
            
            true_dbh = valid_df['actual_dbh'].values
            pred_dbh = valid_df[pred_col].values
            
            # 통계
            slope, intercept, r_value, _, _ = stats.linregress(true_dbh, pred_dbh)
            r_squared = r_value ** 2
            errors = pred_dbh - true_dbh
            rmse = np.sqrt(np.mean(errors**2))
            mae = np.mean(np.abs(errors))
            
            row = idx // 3 + 1
            col = idx % 3 + 1
            
            # y=x 선
            x_range = np.linspace(true_dbh.min()-2, true_dbh.max()+2, 100)
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range,
                mode='lines',
                line=dict(color='gray', width=2, dash='dash'),
                name='y=x',
                showlegend=(idx==0)
            ), row=row, col=col)
            
            # ±RMSE 범위
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range + rmse,
                mode='lines', line=dict(color='lightgray', width=1),
                showlegend=False
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range - rmse,
                mode='lines', line=dict(color='lightgray', width=1),
                fill='tonexty', fillcolor='rgba(200,200,200,0.3)',
                showlegend=False
            ), row=row, col=col)
            
            # 데이터 포인트
            fig.add_trace(go.Scatter(
                x=true_dbh, y=pred_dbh,
                mode='markers',
                marker=dict(size=8, color=colors[idx], opacity=0.7),
                name=ext_name, showlegend=False,
                hovertemplate=f'True: %{{x:.1f}}cm<br>Pred: %{{y:.1f}}cm<extra>{ext_name}</extra>'
            ), row=row, col=col)
            
            # 통계 텍스트
            fig.add_trace(go.Scatter(
                x=[true_dbh.min() + 1], y=[pred_dbh.max() - 1],
                mode='text',
                text=[f'R²={r_squared:.3f}<br>RMSE={rmse:.2f}<br>MAE={mae:.2f}'],
                textposition='bottom right', textfont=dict(size=10),
                showlegend=False
            ), row=row, col=col)
            
            fig.update_xaxes(title_text='True DBH (cm)', row=row, col=col)
            fig.update_yaxes(title_text='Pred DBH (cm)', row=row, col=col)
        
        fig.update_layout(
            title='<b>Mean Slice Range Comparison (IMU+PCA + Augmented Ellipse)</b>',
            width=1200, height=700,
            showlegend=True
        )
        
        fig.write_html(save_path / 'range_comparison.html')
        fig.show()
        
        # 바 차트
        self._plot_range_bar_chart(df, range_names, save_path)
    
    def _plot_range_bar_chart(self, df, range_names, save_path):
        """RMSE 바 차트"""
        
        rmse_data = []
        mae_data = []
        names = []
        
        for ext_key, ext_name in range_names:
            col = f'error_{ext_key}'
            if col in df.columns:
                valid = df[col].dropna()
                if len(valid) > 0:
                    rmse_data.append(np.sqrt(np.mean(valid**2)))
                    mae_data.append(np.mean(np.abs(valid)))
                    names.append(ext_name)
        
        if not names:
            return
        
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            name='RMSE', x=names, y=rmse_data,
            marker_color='steelblue',
            text=[f'{v:.2f}' for v in rmse_data],
            textposition='outside'
        ))
        
        fig.add_trace(go.Bar(
            name='MAE', x=names, y=mae_data,
            marker_color='lightcoral',
            text=[f'{v:.2f}' for v in mae_data],
            textposition='outside'
        ))
        
        fig.update_layout(
            title='<b>Error by Height Range (IMU+PCA + Augmented Ellipse)</b>',
            xaxis_title='Height Range',
            yaxis_title='Error (cm)',
            barmode='group',
            width=800, height=500
        )
        
        fig.write_html(save_path / 'range_bar_chart.html')
        fig.show()


# ============================================================================
# 메인 실행
# ============================================================================

ROOT_PATH = "/workspace/tree"

if not os.path.exists(ROOT_PATH):
    ROOT_PATH = "."

analyzer = VerticalTreeAnalyzer(ROOT_PATH)

print("\n" + "="*70)
print("🌲 Mean Slice Range Comparison (IMU+PCA)")
print("="*70)
print("\n5가지 높이 범위 비교:")
print("  1. 0~100%  (전체)")
print("  2. 10~90%")
print("  3. 20~80%")
print("  4. 30~70%")
print("  5. 40~60%  (중앙)")
print("="*70)

# 분석 실행
df = analyzer.analyze_multiple_frames(
    frame_ids=list(range(3, 52)),
    save_dir='vertical_analysis',
    verbose=False
)

## 범위 변경 (30~70 → 0~100)

In [ ]:
class VerticalTreeAnalyzer:
    """나무를 수직으로 정렬 후 분석"""
    
    def __init__(self, root_dir):
        self.root_dir = Path(root_dir)
        self.loader = TreeDataLoader(root_dir)
    
    def analyze_frame(self, frame_id, alignment_method='imu_then_pca', 
                      height_ratio=0.5, thickness=0.02,
                      extraction_method='all',
                      show_plots=False, save_dir=None, verbose=False,
                      save_dataset=False, dataset_dir=None):
        """단일 프레임 분석
        
        extraction_method:
            - 'slice': IMU+PCA 정렬 (mid_slice)
            - 'no_align': 보정 없음 (mid_slice)
            - 'mean_slice': 30~70% 높이 평균 슬라이스
            - 'all': 모든 방식 비교 (4가지)
        """
        
        # 1. 데이터 로드
        data = self.loader.load_frame(frame_id)
        points_raw = data['points']
        imu = data['imu']
        actual_dbh = data['actual_dbh']
        
        if verbose:
            print(f"Frame {frame_id}: {len(points_raw):,} points, DBH={actual_dbh:.1f}cm")
        
        fits = {}
        results_data = {}
        
        # 데이터셋 저장 경로 설정
        if save_dataset and dataset_dir:
            dataset_path = Path(dataset_dir)
            dataset_path.mkdir(parents=True, exist_ok=True)
        
        # ============================================================
        # 방법 1: IMU+PCA 정렬 - Mid Slice (50%)
        # ============================================================
        if extraction_method in ['slice', 'all']:
            points_aligned, _ = combined_alignment(points_raw, imu, 'imu_then_pca')
            xz_raw, slice_3d = extract_slice_by_ratio(points_aligned, height_ratio, thickness)
            xz_clean, _ = filter_noise_dbscan(xz_raw, eps=0.05, min_samples=5)
            
            fits['mid_slice_imu_pca'] = {
                'circle': fit_circle(xz_clean),
                'ransac_circle': fit_ransac_circle(xz_clean),
                'ellipse': fit_ellipse(xz_clean),
                'ransac_ellipse': fit_ransac_ellipse(xz_clean),
                'augmented_ellipse': fit_with_augmentation(xz_clean)[0]
            }
            results_data['mid_slice_imu_pca'] = {'xz_clean': xz_clean, 'slice_3d': slice_3d}
            
            if save_dataset and dataset_dir and len(xz_clean) > 0:
                save_path = dataset_path / 'mid_slice_imu_pca'
                save_path.mkdir(exist_ok=True)
                xz_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
        
        # ============================================================
        # 방법 2: No Alignment - Mid Slice (50%)
        # ============================================================
        if extraction_method in ['no_align', 'all']:
            noalign_2d_raw, noalign_slice_3d, _ = extract_slice_no_alignment(
                points_raw, height_ratio=height_ratio, thickness=thickness
            )
            noalign_2d_clean, _ = filter_noise_dbscan(noalign_2d_raw, eps=0.05, min_samples=5)
            
            fits['mid_slice_no_align'] = {
                'circle': fit_circle(noalign_2d_clean),
                'ransac_circle': fit_ransac_circle(noalign_2d_clean),
                'ellipse': fit_ellipse(noalign_2d_clean),
                'ransac_ellipse': fit_ransac_ellipse(noalign_2d_clean),
                'augmented_ellipse': fit_with_augmentation(noalign_2d_clean)[0]
            }
            results_data['mid_slice_no_align'] = {'xz_clean': noalign_2d_clean, 'slice_3d': noalign_slice_3d}
            
            if save_dataset and dataset_dir and len(noalign_2d_clean) > 0:
                save_path = dataset_path / 'mid_slice_no_align'
                save_path.mkdir(exist_ok=True)
                noalign_2d_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
        
        # ============================================================
        # 방법 3: IMU+PCA - Mean Slice (30~70% 평균)
        # ============================================================
        if extraction_method in ['mean_slice', 'all']:
            points_aligned, _ = combined_alignment(points_raw, imu, 'imu_then_pca')
            mean_xz_imu = self._extract_mean_slice(points_aligned, height_range=(0.0, 1.0), n_slices=10, thickness=thickness)
            mean_xz_imu_clean, _ = filter_noise_dbscan(mean_xz_imu, eps=0.05, min_samples=5)
            
            fits['mean_slice_imu_pca'] = {
                'circle': fit_circle(mean_xz_imu_clean),
                'ransac_circle': fit_ransac_circle(mean_xz_imu_clean),
                'ellipse': fit_ellipse(mean_xz_imu_clean),
                'ransac_ellipse': fit_ransac_ellipse(mean_xz_imu_clean),
                'augmented_ellipse': fit_with_augmentation(mean_xz_imu_clean)[0]
            }
            results_data['mean_slice_imu_pca'] = {'xz_clean': mean_xz_imu_clean}
            
            if save_dataset and dataset_dir and len(mean_xz_imu_clean) > 0:
                save_path = dataset_path / 'mean_slice_imu_pca'
                save_path.mkdir(exist_ok=True)
                mean_xz_imu_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
        
        # ============================================================
        # 방법 4: No Alignment - Mean Slice (30~70% 평균)
        # ============================================================
        if extraction_method in ['mean_slice', 'all']:
            mean_xz_noalign = self._extract_mean_slice_no_align(points_raw, height_range=(0.0, 1.0), n_slices=10, thickness=thickness)
            mean_xz_noalign_clean, _ = filter_noise_dbscan(mean_xz_noalign, eps=0.05, min_samples=5)
            
            fits['mean_slice_no_align'] = {
                'circle': fit_circle(mean_xz_noalign_clean),
                'ransac_circle': fit_ransac_circle(mean_xz_noalign_clean),
                'ellipse': fit_ellipse(mean_xz_noalign_clean),
                'ransac_ellipse': fit_ransac_ellipse(mean_xz_noalign_clean),
                'augmented_ellipse': fit_with_augmentation(mean_xz_noalign_clean)[0]
            }
            results_data['mean_slice_no_align'] = {'xz_clean': mean_xz_noalign_clean}
            
            if save_dataset and dataset_dir and len(mean_xz_noalign_clean) > 0:
                save_path = dataset_path / 'mean_slice_no_align'
                save_path.mkdir(exist_ok=True)
                mean_xz_noalign_clean.astype(np.float32).tofile(save_path / f'data_{frame_id:04d}.bin')
        
        return {
            'frame_id': frame_id,
            'actual_dbh': actual_dbh,
            'points_raw': points_raw,
            'fits': fits,
            'results_data': results_data
        }
    
    def _extract_mean_slice(self, points_aligned, height_range=(0.0, 1.0), n_slices=10, thickness=0.02):
        """
        30~70% 높이에서 여러 슬라이스를 추출하고 평균 중심으로 정렬하여 합침
        """
        y_vals = points_aligned[:, 1]
        y_min, y_max = y_vals.min(), y_vals.max()
        
        # 슬라이스 높이들
        slice_heights = np.linspace(
            y_min + (y_max - y_min) * height_range[0],
            y_min + (y_max - y_min) * height_range[1],
            n_slices
        )
        
        all_points = []
        for h in slice_heights:
            mask = np.abs(y_vals - h) < thickness / 2
            if mask.sum() > 5:
                slice_pts = points_aligned[mask][:, [0, 2]]  # XZ
                # 각 슬라이스의 중심을 원점으로
                center = slice_pts.mean(axis=0)
                slice_centered = slice_pts - center
                all_points.append(slice_centered)
        
        if len(all_points) == 0:
            return np.array([]).reshape(0, 2)
        
        return np.vstack(all_points)
    
    def _extract_mean_slice_no_align(self, points_raw, height_range=(0.0, 1.0), n_slices=10, thickness=0.02):
        """
        보정 없이 30~70% 높이에서 여러 슬라이스 추출 후 평균
        """
        centroid = points_raw.mean(axis=0)
        centered = points_raw - centroid
        
        y_vals = centered[:, 1]
        y_min, y_max = y_vals.min(), y_vals.max()
        
        slice_heights = np.linspace(
            y_min + (y_max - y_min) * height_range[0],
            y_min + (y_max - y_min) * height_range[1],
            n_slices
        )
        
        all_points = []
        for h in slice_heights:
            mask = np.abs(y_vals - h) < thickness / 2
            if mask.sum() > 5:
                slice_pts = centered[mask][:, [0, 2]]  # XZ
                center = slice_pts.mean(axis=0)
                slice_centered = slice_pts - center
                all_points.append(slice_centered)
        
        if len(all_points) == 0:
            return np.array([]).reshape(0, 2)
        
        return np.vstack(all_points)
    
    def analyze_multiple_frames(self, frame_ids, alignment_method='imu_then_pca', 
                                 extraction_method='all', save_dir='vertical_analysis',
                                 verbose=False, save_dataset=False, dataset_dir=None):
        """여러 프레임 분석
        
        4가지 방식 비교:
        1. mid_slice_imu_pca: IMU+PCA 정렬 + 50% 높이 슬라이스
        2. mid_slice_no_align: 보정 없음 + 50% 높이 슬라이스
        3. mean_slice_imu_pca: IMU+PCA 정렬 + 30~70% 평균 슬라이스
        4. mean_slice_no_align: 보정 없음 + 30~70% 평균 슬라이스
        """
        
        results = []
        save_path = Path(save_dir)
        save_path.mkdir(exist_ok=True)
        
        # 데이터셋 저장 설정
        if save_dataset and dataset_dir:
            ds_path = Path(dataset_dir)
            ds_path.mkdir(parents=True, exist_ok=True)
            print(f"📁 Dataset will be saved to: {dataset_dir}")
        
        print(f"\n{'='*70}")
        print(f"🌲 Analyzing {len(frame_ids)} frames...")
        print(f"{'='*70}")
        
        # 시각화용 샘플 저장
        sample_data = {}
        sample_frame = frame_ids[35]  # 중간 프레임 선택
        
        for fid in tqdm(frame_ids, desc="Analyzing"):
            try:
                result = self.analyze_frame(
                    fid, 
                    alignment_method=alignment_method,
                    extraction_method=extraction_method,
                    show_plots=False, 
                    save_dir=None,
                    verbose=verbose,
                    save_dataset=save_dataset,
                    dataset_dir=dataset_dir
                )
                
                row = {
                    'frame_id': fid,
                    'actual_dbh': result['actual_dbh'],
                }
                
                # 각 방법별 결과 저장
                for ext_key, fit_dict in result['fits'].items():
                    for method, fit in fit_dict.items():
                        if fit:
                            row[f'pred_{ext_key}_{method}'] = fit['dbh']
                            row[f'error_{ext_key}_{method}'] = fit['dbh'] - result['actual_dbh']
                        else:
                            row[f'pred_{ext_key}_{method}'] = np.nan
                            row[f'error_{ext_key}_{method}'] = np.nan
                
                results.append(row)
                
                # 샘플 데이터 저장 (시각화용)
                if fid == sample_frame:
                    sample_data = {
                        'frame_id': fid,
                        'actual_dbh': result['actual_dbh'],
                        'results_data': result['results_data'],
                        'fits': result['fits']
                    }
                
            except Exception as e:
                if verbose:
                    print(f"⚠️ Frame {fid} failed: {e}")
                results.append({'frame_id': fid, 'error': str(e)})
        
        df = pd.DataFrame(results)
        df.to_csv(save_path / 'vertical_analysis_results.csv', index=False)
        
        # 라벨 파일도 데이터셋 폴더에 복사
        if save_dataset and dataset_dir:
            label_df = df[['frame_id', 'actual_dbh']].copy()
            label_df.columns = ['frame', 'dbh']
            label_df.to_csv(Path(dataset_dir) / 'labels.csv', index=False)
            print(f"💾 Labels saved to: {dataset_dir}/labels.csv")
        
        # ============================================================
        # 통계 출력
        # ============================================================
        print(f"\n{'='*70}")
        print("📊 SUMMARY STATISTICS - 4 METHODS COMPARISON")
        print(f"{'='*70}")
        
        fitting_methods = ['augmented_ellipse']
        extraction_methods_list = [
            ('mid_slice_imu_pca', 'Mid Slice + IMU+PCA'),
            ('mid_slice_no_align', 'Mid Slice + No Align'),
            ('mean_slice_imu_pca', 'Mean Slice + IMU+PCA'),
            ('mean_slice_no_align', 'Mean Slice + No Align')
        ]
        
        # 각 추출 방식별 통계
        summary_data = []
        for ext_key, ext_name in extraction_methods_list:
            for method in fitting_methods:
                col = f'error_{ext_key}_{method}'
                if col in df.columns:
                    valid = df[col].dropna()
                    if len(valid) > 0:
                        rmse = np.sqrt(np.mean(valid**2))
                        mae = np.mean(np.abs(valid))
                        bias = np.mean(valid)
                        summary_data.append({
                            'extraction': ext_name,
                            'fitting': method,
                            'rmse': rmse,
                            'mae': mae,
                            'bias': bias
                        })
                        print(f"  {ext_name:<25}: RMSE={rmse:.2f}cm, MAE={mae:.2f}cm, Bias={bias:+.2f}cm")
        
        # 요약 비교
        if len(summary_data) > 0:
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_csv(save_path / 'method_comparison_summary.csv', index=False)
            
            print(f"\n{'='*70}")
            print("🏆 BEST METHOD (by RMSE)")
            print(f"{'='*70}")
            
            best = summary_df.loc[summary_df['rmse'].idxmin()]
            print(f"  {best['extraction']}: RMSE={best['rmse']:.2f}cm")
        
        # ============================================================
        # 비교 그래프 생성
        # ============================================================
        print(f"\n📈 Generating comparison plots...")
        
        self._plot_method_comparison_4(df, extraction_methods_list, fitting_methods, save_path)
        
        # ============================================================
        # Mean Slice 시각화 (IMU+PCA vs No Align)
        # ============================================================
        if sample_data:
            print(f"\n🎨 Visualizing Mean Slice samples (Frame {sample_frame})...")
            self._visualize_mean_slice_samples(sample_data, save_path)
        
        return df
    
    def _plot_method_comparison_4(self, df, extraction_methods, fitting_methods, save_path):
        """4가지 추출 방식 비교 그래프"""
        
        target_fit = 'augmented_ellipse'
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[name for _, name in extraction_methods]
        )
        
        colors = ['blue', 'red', 'green', 'orange']
        
        for idx, (ext_key, ext_name) in enumerate(extraction_methods):
            pred_col = f'pred_{ext_key}_{target_fit}'
            if pred_col not in df.columns:
                continue
            
            valid_df = df[['actual_dbh', pred_col]].dropna()
            if len(valid_df) == 0:
                continue
            
            true_dbh = valid_df['actual_dbh'].values
            pred_dbh = valid_df[pred_col].values
            
            # 통계
            slope, intercept, r_value, _, _ = stats.linregress(true_dbh, pred_dbh)
            r_squared = r_value ** 2
            errors = pred_dbh - true_dbh
            rmse = np.sqrt(np.mean(errors**2))
            mae = np.mean(np.abs(errors))
            
            row = idx // 2 + 1
            col = idx % 2 + 1
            
            # y=x 선
            x_range = np.linspace(true_dbh.min()-2, true_dbh.max()+2, 100)
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range,
                mode='lines',
                line=dict(color='gray', width=2, dash='dash'),
                name='y=x',
                showlegend=(idx==0)
            ), row=row, col=col)
            
            # ±RMSE 범위
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range + rmse,
                mode='lines',
                line=dict(color='lightgray', width=1),
                showlegend=False
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=x_range, y=x_range - rmse,
                mode='lines',
                line=dict(color='lightgray', width=1),
                fill='tonexty',
                fillcolor='rgba(200,200,200,0.3)',
                showlegend=False
            ), row=row, col=col)
            
            # 데이터 포인트
            fig.add_trace(go.Scatter(
                x=true_dbh,
                y=pred_dbh,
                mode='markers',
                marker=dict(size=8, color=colors[idx], opacity=0.7),
                name=ext_name,
                showlegend=False,
                hovertemplate=f'True: %{{x:.1f}}cm<br>Pred: %{{y:.1f}}cm<extra>{ext_name}</extra>'
            ), row=row, col=col)
            
            # 통계 텍스트
            fig.add_trace(go.Scatter(
                x=[true_dbh.min() + 1],
                y=[pred_dbh.max() - 1],
                mode='text',
                text=[f'R²={r_squared:.3f}<br>RMSE={rmse:.2f}<br>MAE={mae:.2f}'],
                textposition='bottom right',
                textfont=dict(size=10),
                showlegend=False
            ), row=row, col=col)
            
            fig.update_xaxes(title_text='True DBH (cm)', row=row, col=col)
            fig.update_yaxes(title_text='Pred DBH (cm)', row=row, col=col)
        
        fig.update_layout(
            title='<b>Method Comparison - Augmented Ellipse</b>',
            width=900,
            height=800,
            showlegend=True
        )
        
        fig.write_html(save_path / 'method_comparison.html')
        fig.show()
        
        # 바 차트
        self._plot_rmse_bar_chart_4(df, extraction_methods, target_fit, save_path)
    
    def _plot_rmse_bar_chart_4(self, df, extraction_methods, target_fit, save_path):
        """RMSE 바 차트"""
        
        rmse_data = []
        mae_data = []
        names = []
        
        for ext_key, ext_name in extraction_methods:
            col = f'error_{ext_key}_{target_fit}'
            if col in df.columns:
                valid = df[col].dropna()
                if len(valid) > 0:
                    rmse_data.append(np.sqrt(np.mean(valid**2)))
                    mae_data.append(np.mean(np.abs(valid)))
                    names.append(ext_name)
        
        if not names:
            return
        
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            name='RMSE',
            x=names,
            y=rmse_data,
            marker_color='steelblue',
            text=[f'{v:.2f}' for v in rmse_data],
            textposition='outside'
        ))
        
        fig.add_trace(go.Bar(
            name='MAE',
            x=names,
            y=mae_data,
            marker_color='lightcoral',
            text=[f'{v:.2f}' for v in mae_data],
            textposition='outside'
        ))
        
        fig.update_layout(
            title='<b>Error Comparison by Method</b>',
            xaxis_title='Method',
            yaxis_title='Error (cm)',
            barmode='group',
            width=800,
            height=500
        )
        
        fig.write_html(save_path / 'error_comparison_bar.html')
        fig.show()
    
    def _visualize_mean_slice_samples(self, sample_data, save_path):
        """Mean Slice 시각화 - IMU+PCA vs No Align"""
        
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=['Mean Slice + IMU+PCA', 'Mean Slice + No Align']
        )
        
        frame_id = sample_data['frame_id']
        actual_dbh = sample_data['actual_dbh']
        
        # IMU+PCA 버전
        if 'mean_slice_imu_pca' in sample_data['results_data']:
            xz_imu = sample_data['results_data']['mean_slice_imu_pca']['xz_clean']
            fit_imu = sample_data['fits'].get('mean_slice_imu_pca', {}).get('augmented_ellipse')
            
            fig.add_trace(go.Scatter(
                x=xz_imu[:, 0] * 100,  # m -> cm
                y=xz_imu[:, 1] * 100,
                mode='markers',
                marker=dict(size=3, color='blue', opacity=0.5),
                name='Points (IMU+PCA)'
            ), row=1, col=1)
            
            # 피팅 결과
            if fit_imu:
                cx, cy = fit_imu['center']
                a, b = fit_imu['a'], fit_imu['b']
                theta_fit = fit_imu['theta']
                t = np.linspace(0, 2*np.pi, 200)
                x_e = a * np.cos(t)
                y_e = b * np.sin(t)
                x_rot = (x_e * np.cos(theta_fit) - y_e * np.sin(theta_fit) + cx) * 100
                y_rot = (x_e * np.sin(theta_fit) + y_e * np.cos(theta_fit) + cy) * 100
                
                fig.add_trace(go.Scatter(
                    x=x_rot, y=y_rot,
                    mode='lines',
                    line=dict(color='red', width=2),
                    name=f'Fit: {fit_imu["dbh"]:.1f}cm'
                ), row=1, col=1)
        
        # No Align 버전
        if 'mean_slice_no_align' in sample_data['results_data']:
            xz_noalign = sample_data['results_data']['mean_slice_no_align']['xz_clean']
            fit_noalign = sample_data['fits'].get('mean_slice_no_align', {}).get('augmented_ellipse')
            
            fig.add_trace(go.Scatter(
                x=xz_noalign[:, 0] * 100,
                y=xz_noalign[:, 1] * 100,
                mode='markers',
                marker=dict(size=3, color='orange', opacity=0.5),
                name='Points (No Align)'
            ), row=1, col=2)
            
            if fit_noalign:
                cx, cy = fit_noalign['center']
                a, b = fit_noalign['a'], fit_noalign['b']
                theta_fit = fit_noalign['theta']
                t = np.linspace(0, 2*np.pi, 200)
                x_e = a * np.cos(t)
                y_e = b * np.sin(t)
                x_rot = (x_e * np.cos(theta_fit) - y_e * np.sin(theta_fit) + cx) * 100
                y_rot = (x_e * np.sin(theta_fit) + y_e * np.cos(theta_fit) + cy) * 100
                
                fig.add_trace(go.Scatter(
                    x=x_rot, y=y_rot,
                    mode='lines',
                    line=dict(color='red', width=2),
                    name=f'Fit: {fit_noalign["dbh"]:.1f}cm'
                ), row=1, col=2)
        
        # 실제 DBH 원 (참고용)
        actual_r = actual_dbh / 2
        theta = np.linspace(0, 2*np.pi, 100)
        
        fig.add_trace(go.Scatter(
            x=actual_r * np.cos(theta),
            y=actual_r * np.sin(theta),
            mode='lines',
            line=dict(color='black', width=2, dash='dash'),
            name=f'Actual: {actual_dbh:.1f}cm'
        ), row=1, col=1)
        
        fig.add_trace(go.Scatter(
            x=actual_r * np.cos(theta),
            y=actual_r * np.sin(theta),
            mode='lines',
            line=dict(color='black', width=2, dash='dash'),
            showlegend=False
        ), row=1, col=2)
        
        fig.update_xaxes(title_text='X (cm)', scaleanchor='y', scaleratio=1, row=1, col=1)
        fig.update_yaxes(title_text='Y (cm)', row=1, col=1)
        fig.update_xaxes(title_text='X (cm)', scaleanchor='y', scaleratio=1, row=1, col=2)
        fig.update_yaxes(title_text='Y (cm)', row=1, col=2)
        
        fig.update_layout(
            title=f'<b>Mean Slice Visualization - Frame {frame_id} (Actual DBH: {actual_dbh:.1f}cm)</b>',
            width=1000,
            height=500,
            showlegend=True
        )
        
        fig.write_html(save_path / 'mean_slice_visualization.html')
        fig.show()


# ============================================================================
# 메인 실행
# ============================================================================

ROOT_PATH = "/workspace/tree"

if not os.path.exists(ROOT_PATH):
    ROOT_PATH = "."

analyzer = VerticalTreeAnalyzer(ROOT_PATH)

print("\n" + "="*70)
print("🌲 VERTICAL TREE ALIGNMENT & DBH ANALYSIS")
print("="*70)
print("\n4가지 방식 비교:")
print("  1. mid_slice_imu_pca   : IMU+PCA 정렬 + 50% 높이 슬라이스")
print("  2. mid_slice_no_align  : 보정 없음 + 50% 높이 슬라이스")
print("  3. mean_slice_imu_pca  : IMU+PCA 정렬 + 0~100% 평균 슬라이스")
print("  4. mean_slice_no_align : 보정 없음 + 0~100% 평균 슬라이스")
print("="*70)

# 데이터셋 저장 경로
DATASET_DIR = f"{ROOT_PATH}/taehee/align_dataset"

# 여러 프레임 분석 - 4가지 방식 비교 + 데이터셋 저장
df = analyzer.analyze_multiple_frames(
    frame_ids=list(range(3, 52)),
    extraction_method='all',
    save_dir='vertical_analysis',
    verbose=False,
    save_dataset=True,
    dataset_dir=DATASET_DIR
)

print(f"\n✅ Dataset saved to: {DATASET_DIR}/")
print("   폴더 구조:")
print("   - mid_slice_imu_pca_2/data_XXXX.bin")
print("   - mid_slice_no_align_2/data_XXXX.bin")
print("   - mean_slice_imu_pca_0to100/data_XXXX.bin")
print("   - mean_slice_no_align_0to100/data_XXXX.bin")
print("   - labels.csv")

## IMU 정렬 → 슬라이스별 PCA 수평정렬 → 다중 슬라이스의 중앙값

> `CombinedTreeAnalyzer`: slice-first IMU + Median 방식 비교

In [ ]:
def extract_raw_slice(points, height_ratio=0.5, thickness=0.02):
    y_vals = points[:, 1]
    y_min, y_max = y_vals.min(), y_vals.max()
    target_height = y_min + (y_max - y_min) * height_ratio
    mask = np.abs(y_vals - target_height) < thickness / 2
    return points[mask]

def align_slice_to_horizontal(slice_points):
    if len(slice_points) < 10:
        return slice_points[:, [0, 2]]
    centroid = slice_points.mean(axis=0)
    centered = slice_points - centroid
    pca = PCA(n_components=3)
    pca.fit(centered)
    normal = pca.components_[2]
    target_normal = np.array([0, 1, 0])
    if np.dot(normal, target_normal) < 0:
        normal = -normal
    rot_matrix = rotation_matrix_from_vectors(normal, target_normal)
    aligned = centered @ rot_matrix.T
    return aligned[:, [0, 2]]

def filter_noise_dbscan(xz, eps=0.05, min_samples=5):
    if len(xz) < min_samples:
        return xz
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(xz)
    labels = clustering.labels_
    unique_labels = set(labels) - {-1}
    if not unique_labels:
        return xz
    largest_cluster = max(unique_labels, key=lambda l: np.sum(labels == l))
    return xz[labels == largest_cluster]

def fit_circle(xz):
    if len(xz) < 3:
        return None
    model = CircleModel()
    if model.estimate(xz):
        return {'method': 'Circle', 'dbh': 2 * model.params[2] * 100}
    return None

def fit_ransac_circle(xz):
    if len(xz) < 10:
        return None
    try:
        model, _ = ransac(xz, CircleModel, min_samples=10, residual_threshold=0.015, max_trials=1000)
        if model:
            return {'method': 'RANSAC Circle', 'dbh': 2 * model.params[2] * 100}
    except:
        pass
    return None

def fit_ellipse(xz):
    if len(xz) < 5:
        return None
    model = EllipseModel()
    if model.estimate(xz):
        a, b = model.params[2], model.params[3]
        return {'method': 'Ellipse', 'dbh': (a + b) * 100}
    return None

def fit_ransac_ellipse(xz):
    if len(xz) < 10:
        return None
    try:
        model, _ = ransac(xz, EllipseModel, min_samples=10, residual_threshold=0.015, max_trials=1000)
        if model:
            a, b = model.params[2], model.params[3]
            return {'method': 'RANSAC Ellipse', 'dbh': (a + b) * 100}
    except:
        pass
    return None

def fit_circle_leastsq(points_2d):
    def calc_residuals(params, points):
        cx, cy, r = params
        distances = np.sqrt((points[:, 0] - cx)**2 + (points[:, 1] - cy)**2)
        return distances - r
    x0 = np.median(points_2d[:, 0])
    y0 = np.median(points_2d[:, 1])
    r0 = np.std(np.linalg.norm(points_2d - [x0, y0], axis=1))
    result = optimize.least_squares(calc_residuals, [x0, y0, r0], args=(points_2d,))
    return result.x[0], result.x[1], abs(result.x[2])

def fit_with_augmentation(points_2d):
    if len(points_2d) < 3:
        return None
    cx, cy, _ = fit_circle_leastsq(points_2d)
    relative = points_2d - np.array([cx, cy])
    augmented = np.vstack([points_2d, -relative + np.array([cx, cy])])
    model = EllipseModel()
    if model.estimate(augmented):
        a, b = model.params[2], model.params[3]
        return {'method': 'Augmented Ellipse', 'dbh': (max(a,b) + min(a,b)) * 100}
    return None

def extract_slice_by_ratio(points, height_ratio=0.5, thickness=0.02):
    y_vals = points[:, 1]
    y_min, y_max = y_vals.min(), y_vals.max()
    target_height = y_min + (y_max - y_min) * height_ratio
    mask = np.abs(y_vals - target_height) < thickness / 2
    return points[mask][:, [0, 2]]

def extract_slice_no_alignment(points, height_ratio=0.5, thickness=0.02):
    centroid = points.mean(axis=0)
    centered = points - centroid
    y_vals = centered[:, 1]
    y_min, y_max = y_vals.min(), y_vals.max()
    target_height = y_min + (y_max - y_min) * height_ratio
    mask = np.abs(y_vals - target_height) < thickness / 2
    return centered[mask][:, [0, 2]]

def extract_mean_slice(points_aligned, height_range=(0.0, 1.0), n_slices=10, thickness=0.02):
    y_vals = points_aligned[:, 1]
    y_min, y_max = y_vals.min(), y_vals.max()
    slice_heights = np.linspace(y_min + (y_max - y_min) * height_range[0],
                                 y_min + (y_max - y_min) * height_range[1], n_slices)
    all_points = []
    for h in slice_heights:
        mask = np.abs(y_vals - h) < thickness / 2
        if mask.sum() > 5:
            slice_pts = points_aligned[mask][:, [0, 2]]
            all_points.append(slice_pts - slice_pts.mean(axis=0))
    return np.vstack(all_points) if all_points else np.array([]).reshape(0, 2)

def extract_mean_slice_no_align(points_raw, height_range=(0.0, 1.0), n_slices=10, thickness=0.02):
    centered = points_raw - points_raw.mean(axis=0)
    y_vals = centered[:, 1]
    y_min, y_max = y_vals.min(), y_vals.max()
    slice_heights = np.linspace(y_min + (y_max - y_min) * height_range[0],
                                 y_min + (y_max - y_min) * height_range[1], n_slices)
    all_points = []
    for h in slice_heights:
        mask = np.abs(y_vals - h) < thickness / 2
        if mask.sum() > 5:
            slice_pts = centered[mask][:, [0, 2]]
            all_points.append(slice_pts - slice_pts.mean(axis=0))
    return np.vstack(all_points) if all_points else np.array([]).reshape(0, 2)


class CombinedTreeAnalyzer:
    def __init__(self, root_dir):
        self.root_dir = Path(root_dir)
        self.loader = TreeDataLoader(root_dir)
    
    def analyze_frame(self, frame_id, thickness=0.02):
        data = self.loader.load_frame(frame_id)
        points_raw = data['points']
        imu = data['imu']
        actual_dbh = data['actual_dbh']
        
        fits = {}
        fitting_methods = [('circle', fit_circle), ('ransac_circle', fit_ransac_circle),
                          ('ellipse', fit_ellipse), ('ransac_ellipse', fit_ransac_ellipse),
                          ('augmented_ellipse', fit_with_augmentation)]
        
        # Method 1: Mid Slice + IMU+PCA
        points_imu_pca = combined_alignment(points_raw, imu)
        xz = filter_noise_dbscan(extract_slice_by_ratio(points_imu_pca, 0.5, thickness))
        fits['mid_slice_imu_pca'] = {name: func(xz) for name, func in fitting_methods}
        
        # Method 2: Mid Slice + No Align
        xz = filter_noise_dbscan(extract_slice_no_alignment(points_raw, 0.5, thickness))
        fits['mid_slice_no_align'] = {name: func(xz) for name, func in fitting_methods}
        
        # Method 3: Mean Slice + IMU+PCA
        xz = filter_noise_dbscan(extract_mean_slice(points_imu_pca, (0.0, 1.0), 10, thickness))
        fits['mean_slice_imu_pca'] = {name: func(xz) for name, func in fitting_methods}
        
        # Method 4: Mean Slice + No Align
        xz = filter_noise_dbscan(extract_mean_slice_no_align(points_raw, (0.0, 1.0), 10, thickness))
        fits['mean_slice_no_align'] = {name: func(xz) for name, func in fitting_methods}
        
        # Method 5: Slice-First IMU (single 50%)
        points_imu = align_with_imu(points_raw, imu)
        slice_3d = extract_raw_slice(points_imu, 0.5, thickness)
        xz = filter_noise_dbscan(align_slice_to_horizontal(slice_3d)) if len(slice_3d) >= 10 else np.array([]).reshape(0,2)
        fits['slice_first_imu'] = {name: func(xz) if len(xz) > 0 else None for name, func in fitting_methods}
        
        # Method 6: Slice-First Median (Code 1 full strategy)
        height_ratios = np.linspace(0.3, 0.7, 25)
        all_dbh = {name: [] for name, _ in fitting_methods}
        
        for h_ratio in height_ratios:
            slice_3d = extract_raw_slice(points_imu, h_ratio, thickness)
            if len(slice_3d) < 10:
                continue
            xz = filter_noise_dbscan(align_slice_to_horizontal(slice_3d))
            if len(xz) < 10:
                continue
            for name, func in fitting_methods:
                result = func(xz)
                if result:
                    all_dbh[name].append(result['dbh'])
        
        fits['slice_first_median'] = {}
        for name in all_dbh:
            dbh_list = all_dbh[name]
            if len(dbh_list) >= 3:
                arr = np.array(dbh_list)
                mean, std = np.mean(arr), np.std(arr)
                if std > 0:
                    arr = arr[np.abs(arr - mean) <= 2 * std]
                fits['slice_first_median'][name] = {'method': name, 'dbh': np.median(arr)} if len(arr) > 0 else None
            elif len(dbh_list) > 0:
                fits['slice_first_median'][name] = {'method': name, 'dbh': np.mean(dbh_list)}
            else:
                fits['slice_first_median'][name] = None
        
        return {'frame_id': frame_id, 'actual_dbh': actual_dbh, 'fits': fits}
    
    def analyze_all_frames(self, frame_ids, save_dir='combined_analysis'):
        results = []
        save_path = Path(save_dir)
        save_path.mkdir(exist_ok=True)
        
        print(f"\n{'='*80}")
        print("COMBINED TREE ANALYSIS - 6 METHODS COMPARISON")
        print(f"{'='*80}")
        print("\nExtraction Methods:")
        print("  1. mid_slice_imu_pca     : IMU+PCA alignment + 50% slice")
        print("  2. mid_slice_no_align    : No alignment + 50% slice")
        print("  3. mean_slice_imu_pca    : IMU+PCA alignment + 0-100% mean slice")
        print("  4. mean_slice_no_align   : No alignment + 0-100% mean slice")
        print("  5. slice_first_imu       : IMU -> per-slice PCA (50%)")
        print("  6. slice_first_median    : IMU -> per-slice PCA -> 25 slices median (Code1)")
        print(f"{'='*80}\n")
        
        for i, fid in enumerate(frame_ids):
            if (i + 1) % 10 == 0 or i == 0:
                print(f"  Processing frame {fid} ({i+1}/{len(frame_ids)})...")
            try:
                result = self.analyze_frame(fid)
                row = {'frame_id': fid, 'actual_dbh': result['actual_dbh']}
                for ext_key, fit_dict in result['fits'].items():
                    for method, fit in fit_dict.items():
                        if fit:
                            row[f'pred_{ext_key}_{method}'] = fit['dbh']
                            row[f'error_{ext_key}_{method}'] = fit['dbh'] - result['actual_dbh']
                        else:
                            row[f'pred_{ext_key}_{method}'] = np.nan
                            row[f'error_{ext_key}_{method}'] = np.nan
                results.append(row)
            except Exception as e:
                print(f"  Frame {fid} failed: {e}")
                results.append({'frame_id': fid, 'error': str(e)})
        
        df = pd.DataFrame(results)
        df.to_csv(save_path / 'combined_analysis_results.csv', index=False)
        self._print_statistics(df, save_path)
        return df
    
    def _print_statistics(self, df, save_path):
        print(f"\n{'='*80}")
        print("SUMMARY STATISTICS")
        print(f"{'='*80}")
        
        extraction_methods = [
            ('mid_slice_imu_pca', 'Mid Slice + IMU+PCA'),
            ('mid_slice_no_align', 'Mid Slice + No Align'),
            ('mean_slice_imu_pca', 'Mean Slice + IMU+PCA'),
            ('mean_slice_no_align', 'Mean Slice + No Align'),
            ('slice_first_imu', 'Slice-First IMU (50%)'),
            ('slice_first_median', 'Slice-First Median (Code1)')
        ]
        fitting_methods = ['circle', 'ransac_circle', 'ellipse', 'ransac_ellipse', 'augmented_ellipse']
        
        summary_data = []
        for ext_key, ext_name in extraction_methods:
            print(f"\n{'-'*60}")
            print(f"  {ext_name}")
            print(f"{'-'*60}")
            for method in fitting_methods:
                col = f'error_{ext_key}_{method}'
                if col in df.columns:
                    valid = df[col].dropna()
                    if len(valid) > 0:
                        rmse = np.sqrt(np.mean(valid**2))
                        mae = np.mean(np.abs(valid))
                        bias = np.mean(valid)
                        summary_data.append({'extraction': ext_name, 'ext_key': ext_key,
                                            'fitting': method, 'rmse': rmse, 'mae': mae, 'bias': bias})
                        print(f"    {method:<20}: RMSE={rmse:6.2f}cm, MAE={mae:6.2f}cm, Bias={bias:+6.2f}cm")
        
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_csv(save_path / 'method_comparison_summary.csv', index=False)
        
        print(f"\n{'='*80}")
        print("BEST METHODS (by RMSE)")
        print(f"{'='*80}")
        for method in fitting_methods:
            method_df = summary_df[summary_df['fitting'] == method]
            if len(method_df) > 0:
                best = method_df.loc[method_df['rmse'].idxmin()]
                print(f"  {method:<20}: {best['extraction']:<30} RMSE={best['rmse']:.2f}cm")
        
        overall_best = summary_df.loc[summary_df['rmse'].idxmin()]
        print(f"\n  OVERALL BEST: {overall_best['extraction']} + {overall_best['fitting']}")
        print(f"     RMSE={overall_best['rmse']:.2f}cm, MAE={overall_best['mae']:.2f}cm")
        
        print(f"\n{'='*80}")
        print("Code1 vs Code2 Comparison (Augmented Ellipse)")
        print(f"{'='*80}")
        aug_df = summary_df[summary_df['fitting'] == 'augmented_ellipse'].sort_values('rmse')
        print(f"\n{'Rank':<4} {'Method':<35} {'RMSE':>8} {'MAE':>8} {'Bias':>10}")
        print("-" * 70)
        for rank, (_, row) in enumerate(aug_df.iterrows(), 1):
            print(f"{rank:<4} {row['extraction']:<35} {row['rmse']:>8.2f} {row['mae']:>8.2f} {row['bias']:>+10.2f}")


ROOT_PATH = "/workspace/tree"
if not os.path.exists(ROOT_PATH):
    ROOT_PATH = "."
analyzer = CombinedTreeAnalyzer(ROOT_PATH)
df = analyzer.analyze_all_frames(frame_ids=list(range(3, 52)), save_dir='combined_analysis')